# Context-Aware Selective Learning for Amsterdam Groundwater Prediction

In this notebook, I test whether transfer learning should always be used for Amsterdam groundwater prediction. My answer is no: some wells benefit from transfer, but other wells are better served by a local model. Because of that, I build a selective-learning framework that decides how much transfer I should use.

I use the processed Amsterdam thesis tables, a `3`-month forecast horizon, and a chronological `60/15/10/15` train/validation/calibration/test split. The main learned-model comparison covers `Site-Hard Expert Selector`, `Deep Context LSTM (Always TL)`, `Context-Aware Selective Learning`, `Deep Context LSTM (No TL)`, and `Random Forest`. I keep `Baseline LSTM (No TL)` and `Baseline LSTM (TL)` as reference LSTM baselines.

My main contribution is not just another LSTM. I separate transferable hydroclimatic signals from Amsterdam-specific context, and then I learn a calibrated gate that can switch between experts instead of using one fixed transfer rule for all wells. I support this with main test results, paired tests, cluster bootstrap, seed stability checks, rolling temporal folds, and ablations.

These baselines are chosen for a focused thesis question. I do not try to benchmark every modern architecture. Instead, I test the decision that matters here: whether Amsterdam forecasting should use no transfer, fixed transfer, or selective transfer under the same data and split design.

In [ ]:
import os
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('NUMEXPR_MAX_THREADS', '1')
os.environ.setdefault('VECLIB_MAXIMUM_THREADS', '1')
import json
import random
import warnings
from copy import deepcopy
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader as TorchDataLoader, TensorDataset
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import grangercausalitytests
from IPython.display import display

warnings.filterwarnings('ignore', category=FutureWarning)

SEED = 42


def set_global_seed(seed: int):
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)


set_global_seed(SEED)
torch.set_num_threads(1)
if hasattr(torch, 'set_num_interop_threads'):
    torch.set_num_interop_threads(1)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 140)


def find_project_root() -> Path:
    root = Path.cwd().resolve()
    if root.name.lower() in {'code', 'data'}:
        root = root.parent
    return root


ROOT = find_project_root()
PROCESSED_DIR = ROOT / 'Data' / 'processed'
MODEL_INPUTS_DIR = PROCESSED_DIR / 'model_inputs'
EDA_DIR = PROCESSED_DIR / 'eda' / 'tables'
TARGET_UNIVERSE_DIR = PROCESSED_DIR / 'eda' / 'target_universe'
RESULTS_DIR = PROCESSED_DIR / 'methodology' / 'tables'
FIGURES_DIR = PROCESSED_DIR / 'methodology' / 'figures'
EDA_TABLE_GROUPS = {
    'overview': {
        'corpus_overview.csv',
        'feature_missingness.csv',
    },
    'correlations': {
        'pooled_lag_correlations.csv',
        'site_best_lag_summary.csv',
        'site_lag_correlations.csv',
    },
    'similarity': {
        'source_target_best_matches.csv',
        'source_target_best_matches_train.csv',
        'source_target_similarity.csv',
        'source_target_similarity_train.csv',
    },
}
EDA_TABLE_DIRS = {group: EDA_DIR / group for group in EDA_TABLE_GROUPS}
EDA_TABLE_PATHS = {
    filename: EDA_TABLE_DIRS[group] / filename
    for group, filenames in EDA_TABLE_GROUPS.items()
    for filename in filenames
}
RESULT_TABLE_GROUPS = {
    'protocol': {
        'protocol_summary.csv',
        'feature_inventory.csv',
        'similarity_summary.csv',
        'transfer_configuration.csv',
    },
    'core_results': {
        'summary_long.csv',
        'summary_wide.csv',
        'gain_vs_no_tl.csv',
        'site_metric_long.csv',
        'site_metric_wide.csv',
        'best_model_by_site.csv',
        'publication_core_comparison.csv',
    },
    'selector': {
        'selector_search.csv',
        'selector_by_site.csv',
        'selector_allocation.csv',
        'selector_feature_importance.csv',
        'site_hard_selector_by_site.csv',
    },
    'transfer': {
        'site_transfer_effects.csv',
        'transfer_effect_summary.csv',
        'cohort_definitions.csv',
        'cohort_assignments.csv',
        'cohort_site_summary.csv',
        'cohort_pooled_summary.csv',
        'negative_transfer_site_summary.csv',
        'negative_transfer_summary.csv',
        'negative_transfer_similarity_bins.csv',
    },
    'error_analysis': {
        'test_error_detail.csv',
        'error_by_anomaly_regime.csv',
        'error_by_signed_anomaly_regime.csv',
        'error_by_calendar_month.csv',
        'site_error_diagnostics.csv',
        'selector_failure_typology.csv',
        'error_driver_correlations.csv',
        'worst_site_error_profile.csv',
    },
    'robustness': {
        'selector_ablation_summary.csv',
        'paired_significance_tests.csv',
        'cluster_bootstrap_significance.csv',
        'label_budget_detail.csv',
        'label_budget_summary.csv',
        'seed_stability_detail.csv',
        'seed_stability_summary.csv',
        'rolling_temporal_folds.csv',
        'rolling_temporal_detail.csv',
        'rolling_temporal_summary.csv',
        'publication_robustness_comparison.csv',
    },
}
RESULT_TABLE_DIRS = {group: RESULTS_DIR / group for group in RESULT_TABLE_GROUPS}
RESULT_TABLE_PATHS = {
    filename: RESULT_TABLE_DIRS[group] / filename
    for group, filenames in RESULT_TABLE_GROUPS.items()
    for filename in filenames
}
for directory in RESULT_TABLE_DIRS.values():
    directory.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cpu')


def eda_table_path(name: str) -> Path:
    if name in EDA_TABLE_PATHS:
        return EDA_TABLE_PATHS[name]
    for directory in EDA_TABLE_DIRS.values():
        candidate = directory / name
        if candidate.exists():
            return candidate
    return EDA_DIR / name


def result_table_path(name: str) -> Path:
    if name in RESULT_TABLE_PATHS:
        return RESULT_TABLE_PATHS[name]
    for directory in RESULT_TABLE_DIRS.values():
        candidate = directory / name
        if candidate.exists():
            return candidate
    return RESULTS_DIR / name


def read_result_csv(name: str) -> pd.DataFrame:
    path = result_table_path(name)
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path)


def methodology_results_available() -> bool:
    required = [
        result_table_path('publication_core_comparison.csv'),
        result_table_path('publication_robustness_comparison.csv'),
        result_table_path('transfer_effect_summary.csv'),
        FIGURES_DIR / 'model_performance_test.png',
    ]
    return all(path.exists() for path in required)


def show_saved_figure(name: str, figsize=(8.5, 4.8)):
    path = FIGURES_DIR / name
    if not path.exists():
        raise FileNotFoundError(path)
    image = plt.imread(path)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(image)
    ax.axis('off')
    plt.tight_layout()
    return fig


print('ROOT:', ROOT)
print('DEVICE:', DEVICE)
print('PROCESSED_DIR:', PROCESSED_DIR)

environment_summary = pd.DataFrame(
    {
        'item': ['project_root', 'processed_dir', 'results_dir', 'figures_dir', 'device'],
        'value': [str(ROOT), str(PROCESSED_DIR), str(RESULTS_DIR), str(FIGURES_DIR), str(DEVICE)],
    }
)
environment_summary


# Data Loading and Processing

## Data Loading

I load the processed Amsterdam tables that I already prepared in the thesis workflow. This keeps the methodology notebook tied to the same fixed dataset that I use in the EDA and in the final results.

The target set is intentionally restricted. I start from Amsterdam BRO monitoring tubes with GLD series, require sufficient metadata span, apply the fixed proximity rule from the data pipeline (`50` meters), and then keep one primary monitoring tube per well for the main analysis. In the current selection summary, the retained download universe must reach at least `2010-01-01` at the start and `2020-01-01` at the end. This process gives `113` selected tubes and `102` final wells. It makes the study reproducible, but it also introduces selection bias: wells with shorter histories, weaker metadata support, or more fragmented records are less likely to enter the final target set.

The source data come from gridded hydroclimatic anomaly cells rather than local urban wells. They give useful regional context, but they do not represent the same measurement process as the Amsterdam observations. This source-target mismatch is one reason why transfer can help in some cases and hurt in others.

In [ ]:
class DataLoader:
    """Loads the processed Amsterdam thesis tables used in the methodology."""

    SIMILARITY_TARGET_SPLITS = ('train',)

    @staticmethod
    def read_parquet(path: Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(path)
        sql = f"SELECT * FROM read_parquet('{path.as_posix()}')"
        return duckdb.sql(sql).df()

    @staticmethod
    def read_csv(path: Path) -> pd.DataFrame:
        if not path.exists():
            raise FileNotFoundError(path)
        return pd.read_csv(path)

    @staticmethod
    def read_csv_optional(path: Path) -> pd.DataFrame:
        if not path.exists():
            return pd.DataFrame()
        return pd.read_csv(path)

    @classmethod
    def similarity_artifact_tag(cls, target_splits: tuple[str, ...] | None = None) -> str:
        splits = target_splits if target_splits is not None else cls.SIMILARITY_TARGET_SPLITS
        cleaned = [str(split).strip().lower() for split in splits if str(split).strip()]
        if not cleaned:
            raise ValueError('At least one target split is required to build similarity artifacts.')
        return '_'.join(dict.fromkeys(cleaned))

    @staticmethod
    def load_selection_summary() -> dict:
        path = TARGET_UNIVERSE_DIR / 'selection_summary.json'
        if not path.exists():
            return {}
        return json.loads(path.read_text())

    @classmethod
    def build_similarity_artifacts(
        cls,
        pretrain_supervised: pd.DataFrame,
        finetune_supervised: pd.DataFrame,
        target_splits: tuple[str, ...] | None = None,
        persist: bool = True,
        artifact_tag_override: str | None = None,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        EDA_DIR.mkdir(parents=True, exist_ok=True)
        target_splits = target_splits if target_splits is not None else cls.SIMILARITY_TARGET_SPLITS
        target_split_set = {str(split) for split in target_splits}
        artifact_tag = artifact_tag_override if artifact_tag_override is not None else cls.similarity_artifact_tag(target_splits)

        source = (
            pretrain_supervised[['cell_id', 'month', 'source_x', 'source_y', 'tsmp_wtda', 'pr_a', 'sm_a']]
            .drop_duplicates(['cell_id', 'month'])
            .copy()
        )
        target_history = finetune_supervised.copy()
        if 'split' not in target_history.columns:
            raise KeyError('finetune_supervised must contain a split column to build train-only similarity artifacts.')
        target_history = target_history[target_history['split'].isin(target_split_set)].copy()
        if target_history.empty:
            raise ValueError(f'No target rows available for similarity splits: {sorted(target_split_set)}')
        target = (
            target_history[['site_id', 'month', 'wtda', 'pr_a', 'sm_a']]
            .drop_duplicates(['site_id', 'month'])
            .copy()
        )
        source['month'] = pd.to_datetime(source['month'])
        target['month'] = pd.to_datetime(target['month'])

        source_groups = {
            int(cell_id): group.sort_values('month').set_index('month')[['tsmp_wtda', 'pr_a', 'sm_a']]
            for cell_id, group in source.groupby('cell_id')
        }
        target_groups = {
            int(site_id): group.sort_values('month').set_index('month')[['wtda', 'pr_a', 'sm_a']]
            for site_id, group in target.groupby('site_id')
        }

        rows = []
        for site_id, target_panel in target_groups.items():
            aligned_target = target_panel.rename(columns={'wtda': 'tsmp_wtda'})
            for cell_id, source_panel in source_groups.items():
                merged = source_panel.join(aligned_target, how='inner', lsuffix='_src', rsuffix='_tgt')
                if merged.empty:
                    continue

                merged = merged[merged['tsmp_wtda_src'].notna() & merged['tsmp_wtda_tgt'].notna()]
                if merged.empty:
                    continue

                source_vec = np.concatenate(
                    [
                        merged['tsmp_wtda_src'].to_numpy(dtype=np.float32),
                        merged['pr_a_src'].to_numpy(dtype=np.float32),
                        merged['sm_a_src'].to_numpy(dtype=np.float32),
                    ]
                )
                target_vec = np.concatenate(
                    [
                        merged['tsmp_wtda_tgt'].to_numpy(dtype=np.float32),
                        merged['pr_a_tgt'].to_numpy(dtype=np.float32),
                        merged['sm_a_tgt'].to_numpy(dtype=np.float32),
                    ]
                )

                source_norm = float(np.linalg.norm(source_vec))
                target_norm = float(np.linalg.norm(target_vec))
                similarity = float((source_vec @ target_vec) / (source_norm * target_norm)) if source_norm > 0 and target_norm > 0 else 0.0
                rows.append((site_id, cell_id, similarity, int(len(merged))))

        if not rows:
            raise ValueError('Cannot rebuild source-target similarity without overlapping groundwater months.')

        pairwise = pd.DataFrame(rows, columns=['site_id', 'cell_id', 'cosine_similarity', 'months_used'])
        pairwise['site_id'] = pairwise['site_id'].astype(np.int64)
        pairwise['cell_id'] = pairwise['cell_id'].astype(np.int64)
        pairwise['cosine_similarity'] = pairwise['cosine_similarity'].astype(np.float32)
        pairwise['months_used'] = pairwise['months_used'].astype(np.int32)

        best_matches = (
            pairwise.sort_values(['site_id', 'cosine_similarity', 'months_used'], ascending=[True, False, False])
            .drop_duplicates('site_id')
            .reset_index(drop=True)
        )

        cell_lookup = (
            source[['cell_id', 'source_x', 'source_y']]
            .drop_duplicates('cell_id')
            .rename(columns={'source_x': 'source_lon', 'source_y': 'source_lat'})
        )
        best_matches = best_matches.merge(cell_lookup, on='cell_id', how='left')

        if persist:
            pairwise.to_csv(eda_table_path(f'source_target_similarity_{artifact_tag}.csv'), index=False)
            best_matches.to_csv(eda_table_path(f'source_target_best_matches_{artifact_tag}.csv'), index=False)
        return best_matches, pairwise

    @classmethod
    def load_processed_tables(cls) -> dict[str, object]:
        pretrain_supervised = cls.read_parquet(MODEL_INPUTS_DIR / 'pretrain_supervised_table.parquet')
        finetune_supervised = cls.read_parquet(MODEL_INPUTS_DIR / 'finetune_supervised_table.parquet')
        artifact_tag = cls.similarity_artifact_tag()
        similarity_dir = EDA_TABLE_DIRS['similarity']
        similarity_parquet_path = similarity_dir / f'source_target_similarity_{artifact_tag}.parquet'
        similarity_csv_path = eda_table_path(f'source_target_similarity_{artifact_tag}.csv')
        best_matches_path = eda_table_path(f'source_target_best_matches_{artifact_tag}.csv')

        if similarity_parquet_path.exists() and best_matches_path.exists():
            best_matches = cls.read_csv(best_matches_path)
            source_target_similarity = cls.read_parquet(similarity_parquet_path)
        elif similarity_csv_path.exists() and best_matches_path.exists():
            best_matches = cls.read_csv(best_matches_path)
            source_target_similarity = cls.read_csv(similarity_csv_path)
        else:
            best_matches, source_target_similarity = cls.build_similarity_artifacts(
                pretrain_supervised=pretrain_supervised,
                finetune_supervised=finetune_supervised,
                target_splits=cls.SIMILARITY_TARGET_SPLITS,
            )

        tables = {
            'pretrain_supervised': pretrain_supervised,
            'finetune_supervised': finetune_supervised,
            'monthly_local': cls.read_parquet(MODEL_INPUTS_DIR / 'monthly_local_wtda.parquet'),
            'baseline_monthly': cls.read_parquet(MODEL_INPUTS_DIR / 'baseline_monthly_table.parquet'),
            'best_matches': best_matches,
            'source_target_similarity': source_target_similarity,
            'pooled_lag_correlations': cls.read_csv_optional(eda_table_path('pooled_lag_correlations.csv')),
            'site_best_lag_summary': cls.read_csv_optional(eda_table_path('site_best_lag_summary.csv')),
            'feature_missingness': cls.read_csv_optional(eda_table_path('feature_missingness.csv')),
            'corpus_overview': cls.read_csv_optional(eda_table_path('corpus_overview.csv')),
            'selection_summary': cls.load_selection_summary(),
            'similarity_target_splits': list(cls.SIMILARITY_TARGET_SPLITS),
        }
        return tables

loader_summary = pd.DataFrame(
    {
        'artifact': [
            'pretrain_supervised_table.parquet',
            'finetune_supervised_table.parquet',
            'summary_wide.csv',
            'publication_core_comparison.csv',
        ],
        'exists': [
            (MODEL_INPUTS_DIR / 'pretrain_supervised_table.parquet').exists(),
            (MODEL_INPUTS_DIR / 'finetune_supervised_table.parquet').exists(),
            result_table_path('summary_wide.csv').exists(),
            result_table_path('publication_core_comparison.csv').exists(),
        ],
        'path': [
            str(MODEL_INPUTS_DIR / 'pretrain_supervised_table.parquet'),
            str(MODEL_INPUTS_DIR / 'finetune_supervised_table.parquet'),
            str(result_table_path('summary_wide.csv')),
            str(result_table_path('publication_core_comparison.csv')),
        ],
    }
)
loader_summary


## Data Processing

I work with `102` target wells and `11,504` supervised site-month rows after anomaly construction, lag generation, and the `3`-month forecast horizon. My source domain contains `512` grid cells and `125,952` supervised source rows for pretraining.

I use monthly anomalies and lagged predictors to make the target and source tables comparable. The shared transfer branch uses the variables that exist in both domains (`tsmp_wtda`, `pr_a`, `sm_a`, their lags, and seasonal encoding). The target context branch uses Amsterdam-specific information (`wtda`, `pumping_m3_month_log1p`, `lon_norm`, and `lat_norm`). This lets me keep transferable information and local context separate.

For preprocessing, I keep only supervised rows where the required lagged inputs and future target are available. In the final modelling table, the shared features, context features, split labels, and `y_target` are complete, so I do not need a later imputation step. I fit each feature scaler on the relevant training windows only and then apply the same transformation to later splits. I scale the forecasting target in the same way, again using target-training statistics only. This avoids information leakage from validation, calibration, or test periods.

I do not pretrain on all source cells. I compute source-target similarity from target training history only, and I use the `maxsim0.60` rule to keep a narrower transfer pool. In the current setup this keeps `88` source cells for pretraining. After I train the two parent deep models, I train the final selective-learning gate on the calibration split. The gate uses similarity summaries, target-history summaries, and recent local state to produce a bounded expert blend instead of one fixed well-level decision.

This design also has a representativeness cost. Monthly aggregation makes the signal more stable, but it can hide short-term extremes. The complete-case supervised table is cleaner for modelling, but shorter and noisier well histories are less represented in the final sample.

In [ ]:
class DataProcessor:
    """Builds thesis-ready inputs from the processed Amsterdam tables."""

    HORIZON = 3
    SPLIT_RATIOS = {'train': 0.60, 'val': 0.15, 'cal': 0.10, 'test': 0.15}
    DEFAULT_SEQUENCE_LENGTH = 9

    SHARED_TRANSFER_FEATURES = [
        'tsmp_wtda', 'pr_a', 'sm_a',
        'tsmp_wtda_lag1', 'tsmp_wtda_lag2', 'tsmp_wtda_lag3',
        'pr_a_lag1', 'pr_a_lag2', 'pr_a_lag3',
        'sm_a_lag1', 'sm_a_lag2', 'sm_a_lag3',
        'month_sin', 'month_cos',
    ]

    TARGET_CONTEXT_FEATURES = [
        'wtda',
        'pumping_m3_month_log1p',
        'lon_norm',
        'lat_norm',
    ]

    TARGET_RF_FEATURES = SHARED_TRANSFER_FEATURES + TARGET_CONTEXT_FEATURES

    @classmethod
    def feature_inventory(cls) -> pd.DataFrame:
        return pd.DataFrame(
            {
                'group': [
                    'shared transfer features',
                    'target context features',
                    'comparison features used by RF and deep models',
                    'bookkeeping / target columns',
                ],
                'features': [
                    ', '.join(cls.SHARED_TRANSFER_FEATURES),
                    ', '.join(cls.TARGET_CONTEXT_FEATURES),
                    ', '.join(cls.TARGET_RF_FEATURES),
                    'site_id, cell_id, month, target_month, split, y_target',
                ],
            }
        )

    @staticmethod
    def protocol_summary(pretrain_df: pd.DataFrame, finetune_df: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame(
            {
                'dataset': ['source pretrain table', 'target finetune table'],
                'rows': [len(pretrain_df), len(finetune_df)],
                'units': [pretrain_df['cell_id'].nunique(), finetune_df['site_id'].nunique()],
                'unit_type': ['source cells', 'target wells'],
                'target_month_start': [pd.to_datetime(pretrain_df['target_month']).min(), pd.to_datetime(finetune_df['target_month']).min()],
                'target_month_end': [pd.to_datetime(pretrain_df['target_month']).max(), pd.to_datetime(finetune_df['target_month']).max()],
                'train_rows': [int((pretrain_df['split'] == 'train').sum()), int((finetune_df['split'] == 'train').sum())],
                'val_rows': [int((pretrain_df['split'] == 'val').sum()), int((finetune_df['split'] == 'val').sum())],
                'cal_rows': [int((pretrain_df['split'] == 'cal').sum()), int((finetune_df['split'] == 'cal').sum())],
                'test_rows': [int((pretrain_df['split'] == 'test').sum()), int((finetune_df['split'] == 'test').sum())],
            }
        )

    @staticmethod
    def build_sequence_windows(
        df: pd.DataFrame,
        group_col: str,
        feature_cols: list[str],
        sequence_length: int = 9,
        target_col: str = 'y_target',
    ):
        windows, targets, meta = [], [], []
        work = df.sort_values([group_col, 'month']).copy()

        for group_value, group_df in work.groupby(group_col):
            group_df = group_df.reset_index(drop=True)
            if len(group_df) < sequence_length:
                continue

            x = group_df[feature_cols].to_numpy(dtype=np.float32)
            y = group_df[target_col].to_numpy(dtype=np.float32)

            for end_idx in range(sequence_length - 1, len(group_df)):
                start_idx = end_idx - sequence_length + 1
                windows.append(x[start_idx : end_idx + 1])
                targets.append(y[end_idx])
                meta.append(
                    {
                        group_col: group_value,
                        'month': group_df.loc[end_idx, 'month'],
                        'target_month': group_df.loc[end_idx, 'target_month'],
                        'split': group_df.loc[end_idx, 'split'],
                    }
                )

        return np.asarray(windows, dtype=np.float32), np.asarray(targets, dtype=np.float32).reshape(-1, 1), pd.DataFrame(meta)

    @staticmethod
    def scale_windows(train_X: np.ndarray, *other_X: np.ndarray):
        mean = train_X.mean(axis=(0, 1), keepdims=True)
        std = train_X.std(axis=(0, 1), keepdims=True)
        std[std == 0] = 1.0
        scaled = [(train_X - mean) / std]
        for arr in other_X:
            scaled.append((arr - mean) / std)
        return scaled, mean, std

display(DataProcessor.feature_inventory())
read_result_csv('protocol_summary.csv')


## Granger Causality Test

I keep this section because it is part of the baseline methodology structure. Here I use it only as a predictive-signal diagnostic, not as strict physical causality evidence.

In [ ]:
def test_granger_causality(df: pd.DataFrame, var1: str, var2: str, max_lags: int = 6) -> pd.DataFrame:
    data = df[[var1, var2]].dropna()
    if len(data) <= max_lags + 5:
        raise ValueError('Not enough observations for the requested Granger test.')

    gc_1_to_2 = grangercausalitytests(data[[var2, var1]], maxlag=max_lags, verbose=False)
    gc_2_to_1 = grangercausalitytests(data[[var1, var2]], maxlag=max_lags, verbose=False)

    return pd.DataFrame(
        {
            'lag': range(1, max_lags + 1),
            f'{var1}_to_{var2}_pvalue': [gc_1_to_2[i][0]['ssr_ftest'][1] for i in range(1, max_lags + 1)],
            f'{var2}_to_{var1}_pvalue': [gc_2_to_1[i][0]['ssr_ftest'][1] for i in range(1, max_lags + 1)],
        }
    )

granger_panel = (
    DataLoader.read_parquet(MODEL_INPUTS_DIR / 'finetune_supervised_table.parquet')[['month', 'wtda', 'tsmp_wtda']]
    .groupby('month', as_index=False)
    .mean()
    .dropna()
)
test_granger_causality(granger_panel, 'tsmp_wtda', 'wtda', max_lags=3).round(4)


# Comparison Models

## ARIMA

I keep a simple univariate ARIMA baseline because it gives me a classical time-series reference with no transfer mechanism and no cross-variable context.

In [ ]:
class ARIMAModel:
    """Simple ARIMA baseline fitted on a one-dimensional target series."""

    def __init__(self, order=(1, 0, 1)):
        self.order = order
        self.model = None
        self.fitted = None

    def train(self, y_train):
        self.model = ARIMA(np.asarray(y_train).flatten(), order=self.order)
        self.fitted = self.model.fit()
        return self.fitted

    def predict(self, steps: int):
        preds = self.fitted.forecast(steps=steps)
        return np.asarray(preds).reshape(-1, 1)

pd.DataFrame(
    {
        'parameter': ['order', 'uses_transfer', 'target_series_only'],
        'value': [str(ARIMAModel().order), 'no', 'yes'],
    }
)


## Random Forest

I keep the Random Forest baseline because it is strong on tabular lag features and gives me a useful non-deep comparison with a different inductive bias. It is not part of the main selective-learning gate; the main gate blends only the no-transfer and always-transfer parent models.

In [ ]:
class RandomForestModel:
    """Random Forest baseline on flattened sequence windows."""

    def __init__(self, n_estimators=80, max_depth=10, min_samples_leaf=2):
        self.params = {
            'n_estimators': n_estimators,
            'max_depth': max_depth,
            'min_samples_leaf': min_samples_leaf,
            'n_jobs': 1,
            'random_state': SEED,
        }
        self.model = RandomForestRegressor(**self.params)

    @staticmethod
    def flatten_windows(X: np.ndarray) -> np.ndarray:
        return X.reshape(X.shape[0], -1)

    def train(self, X_train, y_train):
        self.model.fit(self.flatten_windows(X_train), np.asarray(y_train).flatten())
        return self

    def predict(self, X):
        preds = self.model.predict(self.flatten_windows(X))
        return preds.reshape(-1, 1)

pd.DataFrame(list(RandomForestModel().params.items()), columns=['parameter', 'value'])


# LSTM Models

In [ ]:
def _safe_torch_tensor(array, *, dtype, device=None):
    np_dtype = np.int64 if dtype == torch.long else np.float32
    values = np.asarray(array, dtype=np_dtype).tolist()
    tensor = torch.tensor(values, dtype=dtype)
    if device is not None:
        tensor = tensor.to(device)
    return tensor


class TorchLSTMBackbone(nn.Module):
    """Baseline stacked LSTM backbone: 128 -> 64 -> 32 with dropout."""

    def __init__(self, input_dim, hidden_dims=(128, 64, 32), dropout=0.2):
        super().__init__()
        self.hidden_dims = tuple(int(dim) for dim in hidden_dims)
        dims = (int(input_dim),) + self.hidden_dims
        self.lstm_layers = nn.ModuleList(
            [
                nn.LSTM(
                    input_size=dims[i],
                    hidden_size=dims[i + 1],
                    num_layers=1,
                    batch_first=True,
                )
                for i in range(len(self.hidden_dims))
            ]
        )
        self.dropouts = nn.ModuleList([nn.Dropout(float(dropout)) for _ in self.hidden_dims])
        self.head = nn.Linear(self.hidden_dims[-1], 1)

    def forward(self, x):
        out = x
        for idx, lstm in enumerate(self.lstm_layers):
            out, _ = lstm(out)
            if idx < len(self.lstm_layers) - 1:
                out = self.dropouts[idx](out)
            else:
                out = self.dropouts[idx](out[:, -1, :])
        return self.head(out)


class BaseModel:
    """Base class for baseline PyTorch LSTM variants."""

    def __init__(
        self,
        sequence_length=9,
        batch_size=32,
        epochs=100,
        patience=20,
        lr=1e-3,
        device=DEVICE,
        weight_decay=1e-3,
        scheduler_factor=0.5,
        scheduler_patience=10,
        min_lr=1e-4,
        grad_clip=1.0,
    ):
        self.sequence_length = sequence_length
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.lr = lr
        self.device = device
        self.weight_decay = weight_decay
        self.scheduler_factor = scheduler_factor
        self.scheduler_patience = scheduler_patience
        self.min_lr = min_lr
        self.grad_clip = grad_clip
        self.model = None
        self.history = []

    def build_model(self, input_dim):
        raise NotImplementedError

    def _make_loader(self, X, y, shuffle=False):
        X_t = _safe_torch_tensor(X, dtype=torch.float32)
        y_t = _safe_torch_tensor(y, dtype=torch.float32)
        return TorchDataLoader(TensorDataset(X_t, y_t), batch_size=self.batch_size, shuffle=shuffle)

    def _fit(
        self,
        X_train,
        y_train,
        X_val,
        y_val,
        *,
        lr=None,
        epochs=None,
        patience=None,
        scheduler_factor=None,
        scheduler_patience=None,
    ):
        if self.model is None:
            self.model = self.build_model(X_train.shape[2]).to(self.device)

        train_loader = self._make_loader(X_train, y_train, shuffle=True)
        val_loader = self._make_loader(X_val, y_val, shuffle=False)

        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=float(self.lr if lr is None else lr),
            weight_decay=float(self.weight_decay),
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=float(self.scheduler_factor if scheduler_factor is None else scheduler_factor),
            patience=int(self.scheduler_patience if scheduler_patience is None else scheduler_patience),
            min_lr=float(self.min_lr),
        )
        loss_fn = nn.MSELoss()

        best_val = float('inf')
        best_state = None
        wait = 0
        max_epochs = int(self.epochs if epochs is None else epochs)
        max_wait = int(self.patience if patience is None else patience)

        for epoch in range(1, max_epochs + 1):
            self.model.train()
            train_losses = []
            for xb, yb in train_loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                optimizer.zero_grad()
                pred = self.model(xb)
                loss = loss_fn(pred, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    [p for p in self.model.parameters() if p.requires_grad],
                    max_norm=float(self.grad_clip),
                )
                optimizer.step()
                train_losses.append(float(loss.item()))

            self.model.eval()
            with torch.no_grad():
                val_losses = []
                for xb, yb in val_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    pred = self.model(xb)
                    val_losses.append(float(loss_fn(pred, yb).item()))

            train_loss = float(np.mean(train_losses))
            val_loss = float(np.mean(val_losses))
            self.history.append({'epoch': epoch, 'train_loss': train_loss, 'val_loss': val_loss})
            scheduler.step(val_loss)

            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = deepcopy(self.model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= max_wait:
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)
        return self.history

    def train(self, X_train, y_train, X_val, y_val):
        return self._fit(X_train, y_train, X_val, y_val)

    def predict(self, X):
        self.model.eval()
        X_t = _safe_torch_tensor(X, dtype=torch.float32, device=self.device)
        with torch.no_grad():
            preds = self.model(X_t).cpu().numpy()
        return preds

pd.DataFrame(
    {
        'component': ['TorchLSTMBackbone', 'BaseModel'],
        'role': ['baseline stacked LSTM backbone', 'shared training, early stopping, and prediction logic'],
    }
)


I use the target-only deep context LSTM as my `no transfer` parent. It learns only from Amsterdam wells, so it gives me a clean local deep baseline.

In [ ]:
class LSTMModel(BaseModel):
    """Target-only baseline LSTM over shared inputs only."""

    def __init__(
        self,
        sequence_length=9,
        batch_size=32,
        epochs=100,
        patience=20,
        lr=1e-3,
        hidden_dims=(128, 64, 32),
        dropout=0.2,
        weight_decay=1e-3,
        scheduler_factor=0.5,
        scheduler_patience=10,
        min_lr=1e-4,
    ):
        super().__init__(
            sequence_length=sequence_length,
            batch_size=batch_size,
            epochs=epochs,
            patience=patience,
            lr=lr,
            weight_decay=weight_decay,
            scheduler_factor=scheduler_factor,
            scheduler_patience=scheduler_patience,
            min_lr=min_lr,
        )
        self.hidden_dims = tuple(int(dim) for dim in hidden_dims)
        self.dropout = float(dropout)

    def build_model(self, input_dim):
        return TorchLSTMBackbone(
            input_dim=input_dim,
            hidden_dims=self.hidden_dims,
            dropout=self.dropout,
        )

read_result_csv('transfer_configuration.csv').query("model_family == 'Deep Context LSTM (No TL)'")


I use the transfer version of the same deep context LSTM as my `always transfer` parent. I pretrain its shared encoder on source cells and then fine-tune it on Amsterdam wells. This keeps the comparison focused on transfer initialization rather than on extra model capacity.

In [ ]:
class LSTMTransferLearningModel(LSTMModel):
    """Transfer baseline LSTM with source pretraining and partial freezing."""

    def __init__(
        self,
        sequence_length=9,
        batch_size=32,
        epochs=100,
        patience=20,
        lr=5e-4,
        hidden_dims=(128, 64, 32),
        dropout=0.2,
        weight_decay=1e-3,
        scheduler_factor=0.2,
        scheduler_patience=5,
        min_lr=1e-4,
        n_frozen_layers=2,
        pretrain_lr=1e-3,
        pretrain_epochs=100,
        pretrain_patience=20,
        pretrain_scheduler_factor=0.5,
        pretrain_scheduler_patience=10,
    ):
        super().__init__(
            sequence_length=sequence_length,
            batch_size=batch_size,
            epochs=epochs,
            patience=patience,
            lr=lr,
            hidden_dims=hidden_dims,
            dropout=dropout,
            weight_decay=weight_decay,
            scheduler_factor=scheduler_factor,
            scheduler_patience=scheduler_patience,
            min_lr=min_lr,
        )
        self.pretrained_state = None
        self.n_frozen_layers = int(n_frozen_layers)
        self.pretrain_lr = float(pretrain_lr)
        self.pretrain_epochs = int(pretrain_epochs)
        self.pretrain_patience = int(pretrain_patience)
        self.pretrain_scheduler_factor = float(pretrain_scheduler_factor)
        self.pretrain_scheduler_patience = int(pretrain_scheduler_patience)

    def save_pretrained_state(self):
        self.pretrained_state = deepcopy(self.model.state_dict())

    def load_pretrained_state(self, input_dim):
        self.model = self.build_model(input_dim).to(self.device)
        if self.pretrained_state is None:
            raise ValueError('No pretrained state available. Run source pretraining first.')
        self.model.load_state_dict(self.pretrained_state)
        return self.model

    def freeze_layers(self):
        for layer_idx, lstm in enumerate(self.model.lstm_layers):
            if layer_idx < self.n_frozen_layers:
                for param in lstm.parameters():
                    param.requires_grad = False

    def pretrain(self, X_train, y_train, X_val, y_val):
        self._fit(
            X_train,
            y_train,
            X_val,
            y_val,
            lr=self.pretrain_lr,
            epochs=self.pretrain_epochs,
            patience=self.pretrain_patience,
            scheduler_factor=self.pretrain_scheduler_factor,
            scheduler_patience=self.pretrain_scheduler_patience,
        )
        self.save_pretrained_state()
        return self

    def finetune(self, X_train, y_train, X_val, y_val):
        self.load_pretrained_state(X_train.shape[2])
        self.freeze_layers()
        return self._fit(
            X_train,
            y_train,
            X_val,
            y_val,
            lr=self.lr,
            epochs=self.epochs,
            patience=self.patience,
            scheduler_factor=self.scheduler_factor,
            scheduler_patience=self.scheduler_patience,
        )

read_result_csv('transfer_configuration.csv').query("model_family == 'Deep Context LSTM (Always TL)'")


I keep the MC-Dropout version as a small uncertainty extension. It is not my main thesis result, but it preserves the baseline family and gives me a simple uncertainty-aware variant.

In [ ]:
class MCDropoutLSTM(LSTMModel):
    """Monte Carlo Dropout wrapper around the target-only LSTM."""

    def predict_with_uncertainty(self, X, n_samples=30):
        preds = []
        self.model.train()
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        for _ in range(n_samples):
            with torch.no_grad():
                preds.append(self.model(X_t).cpu().numpy())
        preds = np.stack(preds, axis=0)
        mean_pred = preds.mean(axis=0)
        std_pred = preds.std(axis=0)
        lower = np.percentile(preds, 2.5, axis=0)
        upper = np.percentile(preds, 97.5, axis=0)
        self.model.eval()
        return mean_pred, std_pred, (lower, upper)

pd.DataFrame(
    {
        'parameter': ['uncertainty_samples', 'dropout_based', 'main_result_model'],
        'value': [30, 'yes', 'no'],
    }
)


# Evaluation and Visualizations

## Evaluate Model

I report RMSE, MAE, and `R²`, and I use RMSE as my main ranking metric. I early-stop the parent experts on the validation split. I train the selective-learning gate on the calibration split, and I choose the gate model by grouped out-of-fold mean site RMSE so that wells with longer histories do not dominate the decision.

I also check robustness in several ways. I use observation-level paired tests, site-level paired RMSE tests, site-cluster bootstrap intervals, `5`-seed repeated training, and `3` rolling temporal folds. This helps me show that the result is not just a one-split accident.

This setup has clear trade-offs. A chronological split is more realistic than a random split, but it leaves less data for model fitting. Monthly aggregation makes the learning problem more stable, but it can hide short-lived shocks. Transfer learning can add useful external information, but it can also create negative transfer when the source pattern does not match a local well. Selective learning is more flexible than a fixed transfer rule, but it is also more complex and it requires a separate calibration stage.

I interpret these metrics as predictive evidence, not as proof of groundwater mechanism. Site-specific interpretation can still need domain knowledge, especially when local pumping, urban infrastructure, or subsurface conditions shape unusual behaviour.

In [ ]:
class ModelEvaluator:
    """Computes comparable summary metrics and publication-ready result tables."""

    @staticmethod
    def evaluate_model(y_true, y_pred, model_name='Model'):
        y_true = np.asarray(y_true).flatten()
        y_pred = np.asarray(y_pred).flatten()

        mse = mean_squared_error(y_true, y_pred)
        rmse = float(np.sqrt(mse))
        mae = float(mean_absolute_error(y_true, y_pred))
        r2 = float(r2_score(y_true, y_pred))

        return {
            'model': model_name,
            'rmse': rmse,
            'mae': mae,
            'r2': r2,
            'n': int(len(y_true)),
        }

    @staticmethod
    def to_frame(results: list[dict]) -> pd.DataFrame:
        return pd.DataFrame(results).sort_values('rmse').reset_index(drop=True)

    @staticmethod
    def extract_split_summary(results_df: pd.DataFrame) -> pd.DataFrame:
        out = results_df.copy()
        out['split'] = out['model'].str.extract(r'\((val|cal|test)\)')
        out['model_family'] = out['model'].str.replace(r'\s*\((val|cal|test)\)', '', regex=True)
        return out[['model_family', 'split', 'rmse', 'mae', 'r2', 'n']].sort_values(['split', 'rmse']).reset_index(drop=True)

    @staticmethod
    def wide_summary(summary_df: pd.DataFrame) -> pd.DataFrame:
        metrics = ['rmse', 'mae', 'r2', 'n']
        wide = summary_df.pivot(index='model_family', columns='split', values=metrics)
        wide.columns = [f'{metric}_{split}' for metric, split in wide.columns]
        sort_col = 'rmse_test' if 'rmse_test' in wide.columns else wide.columns[0]
        return wide.reset_index().sort_values(sort_col).reset_index(drop=True)

    @staticmethod
    def gain_vs_baseline(summary_df: pd.DataFrame, baseline='Deep Context LSTM') -> pd.DataFrame:
        rows = []
        for split in sorted(summary_df['split'].dropna().unique()):
            split_df = summary_df[summary_df['split'] == split].copy()
            if baseline not in split_df['model_family'].values:
                continue
            base = split_df[split_df['model_family'] == baseline].iloc[0]
            for _, row in split_df.iterrows():
                if row['model_family'] == baseline:
                    continue
                rows.append({
                    'split': split,
                    'model_family': row['model_family'],
                    'delta_rmse_vs_baseline': row['rmse'] - base['rmse'],
                    'delta_mae_vs_baseline': row['mae'] - base['mae'],
                    'delta_r2_vs_baseline': row['r2'] - base['r2'],
                })
        return pd.DataFrame(rows).sort_values(['split', 'delta_rmse_vs_baseline']).reset_index(drop=True)

    @staticmethod
    def site_level_metrics(model_payloads: dict[str, dict]) -> tuple[pd.DataFrame, pd.DataFrame]:
        rows = []
        for model_name, payload in model_payloads.items():
            meta = payload['test_meta'][['site_id']].copy().reset_index(drop=True)
            df = meta.copy()
            df['y_true'] = np.asarray(payload['test_true']).flatten()
            df['y_pred'] = np.asarray(payload['test_pred']).flatten()
            for site_id, g in df.groupby('site_id'):
                rows.append({
                    'site_id': site_id,
                    'model_family': model_name,
                    'rmse': float(np.sqrt(np.mean((g['y_true'] - g['y_pred']) ** 2))),
                    'mae': float(np.mean(np.abs(g['y_true'] - g['y_pred']))),
                    'n': int(len(g)),
                })
        long_df = pd.DataFrame(rows).sort_values(['site_id', 'rmse']).reset_index(drop=True)
        wide_df = long_df.pivot(index='site_id', columns='model_family', values='rmse').reset_index()
        return long_df, wide_df

    @staticmethod
    def best_model_by_site(site_metric_long: pd.DataFrame) -> pd.DataFrame:
        idx = site_metric_long.groupby('site_id')['rmse'].idxmin()
        best = site_metric_long.loc[idx].copy()
        counts = best.groupby('model_family').size().reset_index(name='best_site_count')
        counts['share_of_sites'] = counts['best_site_count'] / counts['best_site_count'].sum()
        return counts.sort_values(['best_site_count', 'model_family'], ascending=[False, True]).reset_index(drop=True)

    @staticmethod
    def save_table(df: pd.DataFrame, path: Path):
        path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(path, index=False)
        return path

    @staticmethod
    def plot_model_performance(summary_df: pd.DataFrame, fig_dir: Path) -> Path:
        fig_dir.mkdir(parents=True, exist_ok=True)
        test_df = summary_df[summary_df['split'] == 'test'].sort_values('rmse').copy()

        fig, axes = plt.subplots(1, 2, figsize=(13, 5))
        sns.barplot(data=test_df, x='rmse', y='model_family', color='#2a9d8f', ax=axes[0])
        axes[0].set_title('Test RMSE by Model')
        axes[0].set_xlabel('RMSE')
        axes[0].set_ylabel('')

        sns.barplot(data=test_df, x='r2', y='model_family', color='#e76f51', ax=axes[1])
        axes[1].set_title('Test R² by Model')
        axes[1].set_xlabel('R²')
        axes[1].set_ylabel('')

        fig.tight_layout()
        out_path = fig_dir / 'model_performance_test.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    @staticmethod
    def plot_seed_stability(stability_summary: pd.DataFrame, fig_dir: Path) -> Path:
        fig_dir.mkdir(parents=True, exist_ok=True)
        test_df = stability_summary[stability_summary['split'] == 'test'].sort_values('mean_rmse').copy()

        fig, ax = plt.subplots(figsize=(8.5, 5))
        ax.errorbar(
            test_df['mean_rmse'],
            test_df['model_family'],
            xerr=test_df['std_rmse'],
            fmt='o',
            color='#1d3557',
            ecolor='#457b9d',
            elinewidth=1.5,
            capsize=3,
        )
        ax.set_title('Repeated-Seed Stability on Test RMSE')
        ax.set_xlabel('Mean RMSE +/- 1 SD')
        ax.set_ylabel('')
        fig.tight_layout()

        out_path = fig_dir / 'seed_stability_test_rmse.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    @staticmethod
    def plot_rolling_temporal_stability(rolling_summary: pd.DataFrame, fig_dir: Path) -> Path:
        fig_dir.mkdir(parents=True, exist_ok=True)
        test_df = rolling_summary[rolling_summary['split'] == 'test'].sort_values('mean_rmse').copy()

        fig, ax = plt.subplots(figsize=(8.5, 5))
        ax.errorbar(
            test_df['mean_rmse'],
            test_df['model_family'],
            xerr=test_df['std_rmse'],
            fmt='o',
            color='#6a4c93',
            ecolor='#9c89b8',
            elinewidth=1.5,
            capsize=3,
        )
        ax.set_title('Rolling Temporal Robustness on Test RMSE')
        ax.set_xlabel('Mean RMSE +/- 1 SD across rolling folds')
        ax.set_ylabel('')
        fig.tight_layout()

        out_path = fig_dir / 'rolling_temporal_test_rmse.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

display(read_result_csv('summary_wide.csv').round(3))
read_result_csv('publication_core_comparison.csv').round(3)

## Domain Similarity Assessment

I keep a domain-analysis section because source-target compatibility still matters in my thesis. Here similarity is not a single hard threshold. Instead, I use it as one important input to the selective-learning decision.

I do not treat similarity as proof that two locations are physically the same. It is a practical compatibility signal built from overlapping target-training history. This is useful for model selection, but it still has limits because gridded source anomalies and local urban well behaviour are not identical data-generating processes.

In [ ]:
class DomainAnalyzer:
    """Summarizes source-target compatibility diagnostics for selective learning."""

    @staticmethod
    def similarity_summary(best_matches: pd.DataFrame) -> pd.DataFrame:
        return pd.DataFrame(
            {
                'metric': [
                    'best_match_similarity_median',
                    'best_match_similarity_min',
                    'best_match_similarity_max',
                    'best_match_months_median',
                ],
                'value': [
                    best_matches['cosine_similarity'].median(),
                    best_matches['cosine_similarity'].min(),
                    best_matches['cosine_similarity'].max(),
                    best_matches['months_used'].median(),
                ],
            }
        )

    @staticmethod
    def threshold_grid(best_matches: pd.DataFrame, thresholds=(0.20, 0.30, 0.40, 0.50, 0.60)) -> pd.DataFrame:
        rows = []
        for tau in thresholds:
            chosen = best_matches['cosine_similarity'] >= tau
            rows.append(
                {
                    'threshold': tau,
                    'wells_using_transfer': int(chosen.sum()),
                    'wells_without_transfer': int((~chosen).sum()),
                    'share_using_transfer': float(chosen.mean()),
                }
            )
        return pd.DataFrame(rows)

    @staticmethod
    def selector_allocation_summary(blend_df: pd.DataFrame) -> pd.DataFrame:
        if 'selected_expert' in blend_df.columns:
            out = (
                blend_df.groupby('selected_expert', as_index=False)
                .agg(
                    observations=('site_id', 'size'),
                    wells=('site_id', 'nunique'),
                    mean_similarity=('cosine_similarity', 'mean'),
                )
                .rename(columns={'selected_expert': 'path'})
            )
            return out[['path', 'observations', 'wells', 'mean_similarity']]

        out = (
            blend_df.groupby('use_transfer')
            .agg(
                observations=('site_id', 'size'),
                wells=('site_id', 'nunique'),
                mean_similarity=('cosine_similarity', 'mean'),
            )
            .reset_index()
        )
        out['path'] = np.where(out['use_transfer'], 'transfer', 'no_transfer')
        return out[['path', 'observations', 'wells', 'mean_similarity']]

    @staticmethod
    def plot_similarity_distribution(best_matches: pd.DataFrame, fig_dir: Path) -> Path:
        fig_dir.mkdir(parents=True, exist_ok=True)
        fig, ax = plt.subplots(figsize=(8, 4.5))
        sns.histplot(best_matches['cosine_similarity'], bins=40, kde=True, color='#6d597a', ax=ax)
        ax.axvline(best_matches['cosine_similarity'].median(), color='black', lw=1, ls='--')
        ax.set_title('Distribution of Best Source-Target Similarity by Well')
        ax.set_xlabel('Cosine similarity')
        ax.set_ylabel('Count')
        out_path = fig_dir / 'best_match_similarity_distribution.png'
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return out_path

    @staticmethod
    def plot_selector_tradeoff(selector_search: pd.DataFrame, best_label: str, fig_dir: Path) -> Path:
        fig_dir.mkdir(parents=True, exist_ok=True)
        plot_df = selector_search.copy()
        fig, ax1 = plt.subplots(figsize=(8, 4.8))
        sns.barplot(data=plot_df, x='candidate_model', y='mean_site_rmse_oof', color='#2a9d8f', ax=ax1)
        ax1.set_xlabel('')
        ax1.set_ylabel('OOF mean site RMSE', color='#2a9d8f')
        ax1.tick_params(axis='y', labelcolor='#2a9d8f')
        ax1.tick_params(axis='x', rotation=10)
        ax1.set_title(f'Selective Learning Gate Search ({best_label})')

        ax2 = ax1.twinx()
        ax2.plot(np.arange(len(plot_df)), plot_df['share_transfer_oof'], color='#e76f51', marker='o', lw=1.8)
        ax2.set_ylabel('OOF transfer share', color='#e76f51')
        ax2.tick_params(axis='y', labelcolor='#e76f51')

        out_path = fig_dir / 'selector_tradeoff_curve.png'
        fig.tight_layout()
        fig.savefig(out_path, dpi=300, bbox_inches='tight')
        plt.close(fig)
        return out_path

display(read_result_csv('similarity_summary.csv').round(3))
DomainAnalyzer.threshold_grid(DataLoader.read_csv(eda_table_path('source_target_best_matches_train.csv'))).round(3)

## Bootstrapping for Uncertainty Estimation

I keep this section for consistency with the baseline structure. In my thesis, the bootstrap wrapper is a supporting robustness tool rather than the centre of the comparison.

In [ ]:
class BootstrappedLSTM:
    """Simple bootstrap ensemble around the target-only LSTM."""

    def __init__(self, n_bootstrap=5, **model_params):
        self.n_bootstrap = n_bootstrap
        self.model_params = model_params
        self.models = []

    def fit(self, X_train, y_train, X_val, y_val):
        n = len(X_train)
        self.models = []
        for _ in range(self.n_bootstrap):
            idx = np.random.choice(n, size=n, replace=True)
            model = LSTMModel(**self.model_params)
            model.train(X_train[idx], y_train[idx], X_val, y_val)
            self.models.append(model)
        return self

    def predict(self, X):
        preds = np.stack([m.predict(X) for m in self.models], axis=0)
        return preds.mean(axis=0), preds.std(axis=0)

pd.DataFrame(
    {
        'parameter': ['n_bootstrap', 'base_model', 'purpose'],
        'value': [5, 'LSTMModel', 'supporting robustness wrapper'],
    }
)


## Main Implementation

I build the final methodology around three main experts.

1. `Deep Context LSTM (No TL)` learns only from Amsterdam training windows.
2. `Deep Context LSTM (Always TL)` first pretrains on a similarity-filtered source pool and then fine-tunes on Amsterdam.
3. `Random Forest` uses the same overall information budget, but it remains a non-sequential local baseline outside the main selector.

I then add two selector diagnostics.

4. `Context-Aware Selective Learning` uses similarity diagnostics, site history, seasonality, and recent target-state information to learn a bounded observation-level blend inside the expert envelope.
5. `Site-Hard Expert Selector` makes a simpler site-level hard choice and gives me a lower-complexity diagnostic comparison.

I also keep `Baseline LSTM (No TL)` and `Baseline LSTM (TL)` as shared-feature reference models. This lets me compare my context-aware design against the older stacked-LSTM family.

These baselines are appropriate for this thesis because they cover the real alternatives behind my research question: no transfer, fixed transfer, selective transfer, and a strong local non-deep learner. I do not claim that this is a full benchmark of all modern state-of-the-art architectures. For this thesis, the important test is whether selective transfer improves the transfer decision under the same data, horizon, and evaluation design.

In [ ]:
def scale_targets(train_y: np.ndarray, *other_y: np.ndarray):
    mean = train_y.mean(axis=0, keepdims=True)
    std = train_y.std(axis=0, keepdims=True)
    std[std == 0] = 1.0
    scaled = [(train_y - mean) / std]
    for arr in other_y:
        scaled.append((arr - mean) / std)
    return scaled, mean, std


class SequenceEncoder(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 2, dropout: float = 0.2):
        super().__init__()
        effective_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=effective_dropout,
            batch_first=True,
        )

    def forward(self, x):
        out, _ = self.lstm(x)
        return out[:, -1, :]


class ContextAwareTransferBackbone(nn.Module):
    def __init__(self, shared_dim: int, context_dim: int, n_sites: int, shared_hidden: int, context_hidden: int, emb_dim: int = 12):
        super().__init__()
        self.shared_encoder = SequenceEncoder(shared_dim, hidden_dim=shared_hidden, num_layers=2, dropout=0.2)
        self.context_encoder = SequenceEncoder(context_dim, hidden_dim=context_hidden, num_layers=1, dropout=0.0)
        self.site_embedding = nn.Embedding(n_sites, emb_dim)
        self.head = nn.Sequential(
            nn.Linear(shared_hidden + context_hidden + emb_dim + 1, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )

    def forward(self, x_shared, x_context, site_idx):
        h_shared = self.shared_encoder(x_shared)
        h_context = self.context_encoder(x_context)
        h_site = self.site_embedding(site_idx)
        last_wtda = x_context[:, -1, 0:1]
        features = torch.cat([h_shared, h_context, h_site, last_wtda], dim=1)
        return self.head(features)


class SourcePretrainBackbone(nn.Module):
    def __init__(self, shared_dim: int, shared_hidden: int):
        super().__init__()
        self.shared_encoder = SequenceEncoder(shared_dim, hidden_dim=shared_hidden, num_layers=2, dropout=0.2)
        self.head = nn.Sequential(
            nn.Linear(shared_hidden, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 1),
        )

    def forward(self, x_shared):
        return self.head(self.shared_encoder(x_shared))


class NeuralGateNetwork(nn.Module):
    """Lightweight MLP gate used as a deep selector sensitivity check."""

    def __init__(self, input_dim: int, hidden_dims: tuple[int, ...] = (64, 32), dropout: float = 0.1):
        super().__init__()
        dims = (int(input_dim),) + tuple(int(dim) for dim in hidden_dims)
        layers = []
        for idx in range(len(dims) - 1):
            layers.append(nn.Linear(dims[idx], dims[idx + 1]))
            layers.append(nn.ReLU())
            if float(dropout) > 0:
                layers.append(nn.Dropout(float(dropout)))
        layers.append(nn.Linear(dims[-1], 1))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return torch.sigmoid(self.network(x))


class NeuralGateRegressor:
    """Small tabular MLP with early stopping for selector calibration."""

    def __init__(
        self,
        input_dim: int,
        seed: int,
        hidden_dims: tuple[int, ...] = (64, 32),
        dropout: float = 0.1,
        lr: float = 1e-3,
        epochs: int = 180,
        patience: int = 18,
        batch_size: int = 128,
        weight_decay: float = 1e-4,
        val_fraction: float = 0.15,
    ):
        self.input_dim = int(input_dim)
        self.seed = int(seed)
        self.hidden_dims = tuple(int(dim) for dim in hidden_dims)
        self.dropout = float(dropout)
        self.lr = float(lr)
        self.epochs = int(epochs)
        self.patience = int(patience)
        self.batch_size = int(batch_size)
        self.weight_decay = float(weight_decay)
        self.val_fraction = float(val_fraction)
        self.model = None
        self.x_mean_ = None
        self.x_std_ = None

    @staticmethod
    def _weighted_mse(pred, target, weight):
        pred = pred.reshape(-1, 1)
        target = target.reshape(-1, 1)
        weight = weight.reshape(-1, 1)
        return torch.mean(weight * torch.square(pred - target))

    def fit(self, X, y, sample_weight=None):
        set_global_seed(self.seed)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32).reshape(-1, 1)
        if sample_weight is None:
            sample_weight = np.ones(len(X), dtype=np.float32)
        sample_weight = np.asarray(sample_weight, dtype=np.float32).reshape(-1, 1)

        self.x_mean_ = X.mean(axis=0, keepdims=True)
        self.x_std_ = X.std(axis=0, keepdims=True)
        self.x_std_[self.x_std_ < 1e-6] = 1.0
        X_scaled = (X - self.x_mean_) / self.x_std_

        n = len(X_scaled)
        if n == 0:
            raise ValueError('NeuralGateRegressor received an empty design matrix.')
        if n == 1:
            train_idx = np.asarray([0], dtype=np.int64)
            val_idx = np.asarray([0], dtype=np.int64)
        else:
            rng = np.random.default_rng(self.seed)
            perm = rng.permutation(n)
            val_size = max(1, int(round(n * self.val_fraction)))
            val_size = min(val_size, n - 1)
            val_idx = np.sort(perm[:val_size])
            train_idx = np.sort(perm[val_size:])
            if train_idx.size == 0:
                train_idx = val_idx.copy()

        X_train = _safe_torch_tensor(X_scaled[train_idx], dtype=torch.float32)
        y_train = _safe_torch_tensor(y[train_idx], dtype=torch.float32)
        w_train = _safe_torch_tensor(sample_weight[train_idx], dtype=torch.float32)
        X_val = _safe_torch_tensor(X_scaled[val_idx], dtype=torch.float32)
        y_val = _safe_torch_tensor(y[val_idx], dtype=torch.float32)
        w_val = _safe_torch_tensor(sample_weight[val_idx], dtype=torch.float32)

        train_loader = TorchDataLoader(
            TensorDataset(X_train, y_train, w_train),
            batch_size=min(self.batch_size, max(1, len(train_idx))),
            shuffle=True,
        )
        val_loader = TorchDataLoader(
            TensorDataset(X_val, y_val, w_val),
            batch_size=min(self.batch_size, max(1, len(val_idx))),
            shuffle=False,
        )

        self.model = NeuralGateNetwork(
            input_dim=self.input_dim,
            hidden_dims=self.hidden_dims,
            dropout=self.dropout,
        ).to(DEVICE)
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr=self.lr,
            weight_decay=self.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=max(3, self.patience // 3),
            min_lr=1e-5,
        )

        best_val = float('inf')
        best_state = deepcopy(self.model.state_dict())
        wait = 0

        for _ in range(self.epochs):
            self.model.train()
            for xb, yb, wb in train_loader:
                xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
                optimizer.zero_grad()
                pred = self.model(xb)
                loss = self._weighted_mse(pred, yb, wb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()

            self.model.eval()
            val_losses = []
            with torch.no_grad():
                for xb, yb, wb in val_loader:
                    xb, yb, wb = xb.to(DEVICE), yb.to(DEVICE), wb.to(DEVICE)
                    pred = self.model(xb)
                    val_losses.append(float(self._weighted_mse(pred, yb, wb).item()))

            val_loss = float(np.mean(val_losses))
            scheduler.step(val_loss)

            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = deepcopy(self.model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= self.patience:
                    break

        self.model.load_state_dict(best_state)
        return self

    def predict(self, X):
        if self.model is None or self.x_mean_ is None or self.x_std_ is None:
            raise ValueError('NeuralGateRegressor must be fitted before calling predict.')
        X = np.asarray(X, dtype=np.float32)
        X_scaled = (X - self.x_mean_) / self.x_std_
        X_t = _safe_torch_tensor(X_scaled, dtype=torch.float32, device=DEVICE)
        self.model.eval()
        with torch.no_grad():
            pred = self.model(X_t).cpu().numpy().reshape(-1)
        return pred


class LSTMTransferLearningImplementation:
    """Main implementation class for context-aware selective learning."""

    NO_TL_NAME = 'Deep Context LSTM (No TL)'
    ALWAYS_TL_NAME = 'Deep Context LSTM (Always TL)'
    SELECTIVE_TL_NAME = 'Context-Aware Selective Learning'
    SITE_HARD_TL_NAME = 'Site-Hard Expert Selector'
    SITE_SELECTOR_MIN_NO_TL_MARGIN = 0.02
    SITE_SELECTOR_MIN_OVERLAP_MONTHS = 72
    BASELINE_NO_TL_NAME = 'Baseline LSTM (No TL)'
    BASELINE_TL_NAME = 'Baseline LSTM (TL)'
    SELECTOR_MAIN_KEY = 'context_similarity_state'
    SELECTOR_FULL_KEY = 'full'

    def __init__(self, root: Path | None = None, sequence_length: int = 9, seed: int = SEED):
        self.root = Path(root) if root is not None else ROOT
        self.sequence_length = sequence_length
        self.seed = int(seed)
        self.tables = None
        self.results = {}
        self.figure_dir = self.root / 'Data' / 'processed' / 'methodology' / 'figures'
        self.table_dir = self.root / 'Data' / 'processed' / 'methodology' / 'tables'
        self.train_window_budget = None

        self.parent_config = {
            'shared_hidden': 64,
            'context_hidden': 28,
            'target_lr': 7e-4,
            'target_epochs': 28,
            'target_patience': 6,
        }
        self.transfer_config = {
            'source_mode': 'maxsim0.60',
            'pretrain_lr': 1e-3,
            'pretrain_epochs': 12,
            'pretrain_patience': 3,
        }
        self.baseline_config = {
            'source_mode': 'maxsim0.60',
            'hidden_dims': (128, 64, 32),
            'dropout': 0.2,
            'weight_decay': 1e-3,
            'batch_size': 256,
            'target_lr': 1e-3,
            'target_epochs': 25,
            'target_patience': 5,
            'target_scheduler_factor': 0.5,
            'target_scheduler_patience': 2,
            'tl_lr': 5e-4,
            'tl_epochs': 20,
            'tl_patience': 4,
            'tl_scheduler_factor': 0.2,
            'tl_scheduler_patience': 2,
            'pretrain_lr': 1e-3,
            'pretrain_epochs': 8,
            'pretrain_patience': 3,
            'pretrain_scheduler_factor': 0.5,
            'pretrain_scheduler_patience': 2,
            'n_frozen_layers': 2,
        }
        self.selector_config = {
            'main_feature_set': self.SELECTOR_MAIN_KEY,
            'neural_hidden_dims': (64, 32),
            'neural_dropout': 0.10,
            'neural_lr': 1e-3,
            'neural_epochs': 180,
            'neural_patience': 18,
            'neural_batch_size': 128,
            'neural_weight_decay': 1e-4,
        }
        self.selector_split = 'cal'

    def load_data(self):
        self.tables = DataLoader.load_processed_tables()
        return self.tables

    @staticmethod
    def _mask(meta: pd.DataFrame, split: str) -> np.ndarray:
        return meta['split'].eq(split).to_numpy()

    @staticmethod
    def _budget_mask_from_meta(
        meta: pd.DataFrame,
        split: str = 'train',
        max_windows_per_site: int | None = None,
    ) -> np.ndarray:
        base_mask = meta['split'].eq(split).to_numpy()
        if max_windows_per_site is None:
            return base_mask

        max_windows_per_site = int(max_windows_per_site)
        if max_windows_per_site <= 0:
            return np.zeros(len(meta), dtype=bool)

        base_idx = np.flatnonzero(base_mask)
        if base_idx.size == 0:
            return base_mask

        budget_meta = meta.loc[base_idx, ['site_id', 'target_month']].copy()
        budget_meta['row_idx'] = base_idx
        budget_meta['target_month'] = pd.to_datetime(budget_meta['target_month'])

        keep_idx = (
            budget_meta.sort_values(['site_id', 'target_month'])
            .groupby('site_id', group_keys=False)
            .tail(max_windows_per_site)['row_idx']
            .to_numpy(dtype=np.int64)
        )
        out = np.zeros(len(meta), dtype=bool)
        out[keep_idx] = True
        return out

    def _target_train_mask(self, meta: pd.DataFrame | None = None) -> np.ndarray:
        target_meta = self.results['target_sequences']['meta'] if meta is None else meta
        return self._budget_mask_from_meta(
            target_meta,
            split='train',
            max_windows_per_site=self.train_window_budget,
        )

    def _inherit_trial_configuration(self, trial: 'LSTMTransferLearningImplementation') -> 'LSTMTransferLearningImplementation':
        trial.parent_config = deepcopy(self.parent_config)
        trial.transfer_config = deepcopy(self.transfer_config)
        trial.baseline_config = deepcopy(self.baseline_config)
        return trial

    @staticmethod
    def _make_loader(arrays: list[np.ndarray], y: np.ndarray, batch_size: int = 128, shuffle: bool = False):
        tensors = []
        for arr in arrays:
            if np.issubdtype(arr.dtype, np.integer):
                tensors.append(_safe_torch_tensor(arr, dtype=torch.long))
            else:
                tensors.append(_safe_torch_tensor(arr, dtype=torch.float32))
        tensors.append(_safe_torch_tensor(y, dtype=torch.float32))
        return TorchDataLoader(TensorDataset(*tensors), batch_size=batch_size, shuffle=shuffle)

    def _fit_model(self, model: nn.Module, train_loader, val_loader, lr: float, epochs: int, patience: int):
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(trainable_params, lr=lr)
        loss_fn = nn.MSELoss()
        best_val = float('inf')
        best_state = None
        wait = 0

        for _ in range(epochs):
            model.train()
            for batch in train_loader:
                *xs, yb = batch
                xs = [x.to(DEVICE) for x in xs]
                yb = yb.to(DEVICE)
                optimizer.zero_grad()
                pred = model(*xs)
                loss = loss_fn(pred, yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
                optimizer.step()

            model.eval()
            val_losses = []
            with torch.no_grad():
                for batch in val_loader:
                    *xs, yb = batch
                    xs = [x.to(DEVICE) for x in xs]
                    yb = yb.to(DEVICE)
                    pred = model(*xs)
                    val_losses.append(loss_fn(pred, yb).item())

            val_loss = float(np.mean(val_losses))
            if val_loss < best_val - 1e-5:
                best_val = val_loss
                best_state = deepcopy(model.state_dict())
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break

        if best_state is not None:
            model.load_state_dict(best_state)
        return model

    @staticmethod
    def _predict(model: nn.Module, arrays: list[np.ndarray], y_mean: np.ndarray, y_std: np.ndarray) -> np.ndarray:
        model.eval()
        tensors = []
        for arr in arrays:
            if np.issubdtype(arr.dtype, np.integer):
                tensors.append(_safe_torch_tensor(arr, dtype=torch.long, device=DEVICE))
            else:
                tensors.append(_safe_torch_tensor(arr, dtype=torch.float32, device=DEVICE))
        with torch.no_grad():
            pred = model(*tensors).cpu().numpy()
        return pred * y_std + y_mean

    @staticmethod
    def _site_level_rmse(y_true: np.ndarray, y_pred: np.ndarray, meta: pd.DataFrame, value_name: str) -> pd.DataFrame:
        frame = meta[['site_id']].copy().reset_index(drop=True)
        frame['y_true'] = np.asarray(y_true).flatten()
        frame['y_pred'] = np.asarray(y_pred).flatten()
        frame['sq_err'] = (frame['y_true'] - frame['y_pred']) ** 2
        out = frame.groupby('site_id', as_index=False)['sq_err'].mean()
        out[value_name] = np.sqrt(out['sq_err'])
        out = out.drop(columns='sq_err')
        return out

    @staticmethod
    def _site_level_counts(meta: pd.DataFrame, value_name: str) -> pd.DataFrame:
        out = meta.groupby('site_id', as_index=False).size()
        return out.rename(columns={'size': value_name})

    def _select_source_cells(self, mode: str) -> tuple[list[int], pd.DataFrame]:
        similarity = self.tables['source_target_similarity'][['site_id', 'cell_id', 'cosine_similarity']].copy()
        cell_summary = (
            similarity.groupby('cell_id', as_index=False)['cosine_similarity']
            .max()
            .rename(columns={'cosine_similarity': 'max_cosine_similarity'})
        )

        if mode == 'all':
            selected = cell_summary['cell_id'].sort_values().tolist()
        elif mode.startswith('maxsim'):
            tau = float(mode.replace('maxsim', ''))
            selected = cell_summary.loc[cell_summary['max_cosine_similarity'] >= tau, 'cell_id'].sort_values().tolist()
        elif mode.startswith('top'):
            k = int(mode.replace('top', ''))
            ranked = similarity.sort_values(['site_id', 'cosine_similarity'], ascending=[True, False])
            selected = sorted(ranked.groupby('site_id').head(k)['cell_id'].unique().tolist())
        else:
            raise ValueError(f'Unknown source_mode: {mode}')

        selected_summary = cell_summary[cell_summary['cell_id'].isin(selected)].copy().reset_index(drop=True)
        return selected, selected_summary

    def summarize_protocol(self):
        pretrain_df = self.tables['pretrain_supervised']
        finetune_df = self.tables['finetune_supervised']
        return {
            'selection_summary': self.tables['selection_summary'],
            'protocol_summary': DataProcessor.protocol_summary(pretrain_df, finetune_df),
            'feature_inventory': DataProcessor.feature_inventory(),
            'similarity_summary': DomainAnalyzer.similarity_summary(self.tables['best_matches']),
            'transfer_configuration': self.results['transfer_configuration'],
        }

    def prepare_datasets(self):
        source_df = self.tables['pretrain_supervised'].copy()
        target_df = self.tables['finetune_supervised'].copy()

        X_src_shared, y_src, meta_src = DataProcessor.build_sequence_windows(
            source_df, 'cell_id', DataProcessor.SHARED_TRANSFER_FEATURES, sequence_length=self.sequence_length
        )
        X_tgt_shared, y_tgt, meta_tgt = DataProcessor.build_sequence_windows(
            target_df, 'site_id', DataProcessor.SHARED_TRANSFER_FEATURES, sequence_length=self.sequence_length
        )
        X_tgt_context, _, _ = DataProcessor.build_sequence_windows(
            target_df, 'site_id', DataProcessor.TARGET_CONTEXT_FEATURES, sequence_length=self.sequence_length
        )
        X_rf, y_rf, meta_rf = DataProcessor.build_sequence_windows(
            target_df, 'site_id', DataProcessor.TARGET_RF_FEATURES, sequence_length=self.sequence_length
        )

        src_train = self._mask(meta_src, 'train')
        tgt_train = self._target_train_mask(meta_tgt)
        if tgt_train.sum() == 0:
            raise ValueError('The requested target train-window budget leaves no supervised target windows.')

        (_, X_tgt_shared_scaled), shared_mean, shared_std = DataProcessor.scale_windows(
            X_src_shared[src_train],
            X_tgt_shared,
        )
        X_src_shared_scaled = (X_src_shared - shared_mean) / shared_std
        (_, X_tgt_context_scaled), context_mean, context_std = DataProcessor.scale_windows(
            X_tgt_context[tgt_train],
            X_tgt_context,
        )
        (_, y_tgt_scaled), y_mean, y_std = scale_targets(
            y_tgt[tgt_train],
            y_tgt,
        )
        y_src_scaled = (y_src - y_mean) / y_std

        site_ids = sorted(target_df['site_id'].unique().tolist())
        site_to_idx = {site_id: idx for idx, site_id in enumerate(site_ids)}
        site_idx = np.asarray([site_to_idx[site_id] for site_id in meta_tgt['site_id']], dtype=np.int64)
        effective_train_window_summary = (
            pd.DataFrame({'site_id': site_ids})
            .merge(
                meta_tgt.loc[tgt_train, ['site_id']]
                .groupby('site_id', as_index=False)
                .size()
                .rename(columns={'size': 'train_windows_used'}),
                on='site_id',
                how='left',
            )
        )
        effective_train_window_summary['train_windows_used'] = (
            effective_train_window_summary['train_windows_used'].fillna(0).astype(np.int32)
        )

        self.results['source_sequences'] = {
            'X_shared': X_src_shared_scaled,
            'y': y_src,
            'y_scaled': y_src_scaled,
            'meta': meta_src.reset_index(drop=True),
        }
        self.results['target_sequences'] = {
            'X_shared': X_tgt_shared_scaled,
            'X_context': X_tgt_context_scaled,
            'X_context_raw': X_tgt_context,
            'X_rf': X_rf,
            'y': y_tgt,
            'y_scaled': y_tgt_scaled,
            'meta': meta_tgt.reset_index(drop=True),
            'site_idx': site_idx,
            'site_ids': site_ids,
            'site_to_idx': site_to_idx,
            'y_mean': y_mean,
            'y_std': y_std,
            'context_mean': context_mean,
            'context_std': context_std,
            'shared_mean': shared_mean,
            'shared_std': shared_std,
        }
        self.results['effective_train_window_summary'] = effective_train_window_summary
        self.results['effective_train_window_budget'] = (
            int(self.train_window_budget) if self.train_window_budget is not None else np.nan
        )
        self.results['effective_train_windows_total'] = int(tgt_train.sum())

        selected_source_cells, selected_cell_summary = self._select_source_cells(self.transfer_config['source_mode'])
        selected_source_mask = self.results['source_sequences']['meta']['cell_id'].isin(selected_source_cells).to_numpy()
        selected_train_windows = int((self._mask(self.results['source_sequences']['meta'], 'train') & selected_source_mask).sum())
        selected_val_windows = int((self._mask(self.results['source_sequences']['meta'], 'val') & selected_source_mask).sum())
        baseline_source_cells, _ = self._select_source_cells(self.baseline_config['source_mode'])
        baseline_source_mask = self.results['source_sequences']['meta']['cell_id'].isin(baseline_source_cells).to_numpy()
        baseline_train_windows = int((self._mask(self.results['source_sequences']['meta'], 'train') & baseline_source_mask).sum())
        baseline_val_windows = int((self._mask(self.results['source_sequences']['meta'], 'val') & baseline_source_mask).sum())

        self.results['selected_source_cells'] = selected_source_cells
        self.results['selected_source_summary'] = selected_cell_summary
        self.results['best_match_lookup'] = self.tables['best_matches'][['site_id', 'cell_id', 'cosine_similarity', 'months_used']].drop_duplicates('site_id')
        self.results['transfer_configuration'] = pd.DataFrame(
            [
                {
                    'model_family': self.NO_TL_NAME,
                    'architecture': 'context_backbone_64_28',
                    'transfer_mode': 'no_tl',
                    'source_mode': 'none',
                    'source_cells': 0,
                    'source_train_windows': 0,
                    'source_val_windows': 0,
                    'target_train_windows': int(tgt_train.sum()),
                    'target_train_window_budget': self.train_window_budget,
                    'dropout': 0.2,
                    'weight_decay': np.nan,
                    **self.parent_config,
                    'pretrain_lr': np.nan,
                    'pretrain_epochs': 0,
                    'pretrain_patience': 0,
                },
                {
                    'model_family': self.ALWAYS_TL_NAME,
                    'architecture': 'context_backbone_64_28',
                    'transfer_mode': 'always_tl',
                    'source_mode': self.transfer_config['source_mode'],
                    'source_cells': len(selected_source_cells),
                    'source_train_windows': selected_train_windows,
                    'source_val_windows': selected_val_windows,
                    'target_train_windows': int(tgt_train.sum()),
                    'target_train_window_budget': self.train_window_budget,
                    'dropout': 0.2,
                    'weight_decay': np.nan,
                    **self.parent_config,
                    **self.transfer_config,
                },
                {
                    'model_family': self.BASELINE_NO_TL_NAME,
                    'architecture': 'baseline_stacked_lstm_128_64_32',
                    'transfer_mode': 'no_tl',
                    'source_mode': 'none',
                    'source_cells': 0,
                    'source_train_windows': 0,
                    'source_val_windows': 0,
                    'target_train_windows': int(tgt_train.sum()),
                    'target_train_window_budget': self.train_window_budget,
                    'shared_hidden': self.baseline_config['hidden_dims'][0],
                    'context_hidden': np.nan,
                    'dropout': self.baseline_config['dropout'],
                    'weight_decay': self.baseline_config['weight_decay'],
                    'target_lr': self.baseline_config['target_lr'],
                    'target_epochs': self.baseline_config['target_epochs'],
                    'target_patience': self.baseline_config['target_patience'],
                    'pretrain_lr': np.nan,
                    'pretrain_epochs': 0,
                    'pretrain_patience': 0,
                },
                {
                    'model_family': self.BASELINE_TL_NAME,
                    'architecture': 'baseline_stacked_lstm_128_64_32',
                    'transfer_mode': 'always_tl',
                    'source_mode': self.baseline_config['source_mode'],
                    'source_cells': len(baseline_source_cells),
                    'source_train_windows': baseline_train_windows,
                    'source_val_windows': baseline_val_windows,
                    'target_train_windows': int(tgt_train.sum()),
                    'target_train_window_budget': self.train_window_budget,
                    'shared_hidden': self.baseline_config['hidden_dims'][0],
                    'context_hidden': np.nan,
                    'dropout': self.baseline_config['dropout'],
                    'weight_decay': self.baseline_config['weight_decay'],
                    'target_lr': self.baseline_config['tl_lr'],
                    'target_epochs': self.baseline_config['tl_epochs'],
                    'target_patience': self.baseline_config['tl_patience'],
                    'pretrain_lr': self.baseline_config['pretrain_lr'],
                    'pretrain_epochs': self.baseline_config['pretrain_epochs'],
                    'pretrain_patience': self.baseline_config['pretrain_patience'],
                },
            ]
        )
        return self.results

    def run_random_forest(self):
        set_global_seed(self.seed)
        payload = self.results['target_sequences']
        X_rf = payload['X_rf']
        y = payload['y']
        meta = payload['meta']

        train_mask = self._target_train_mask(meta)
        val_mask = self._mask(meta, 'val')
        cal_mask = self._mask(meta, 'cal')
        test_mask = self._mask(meta, 'test')

        model = RandomForestModel(n_estimators=80, max_depth=10, min_samples_leaf=2)
        model.params['random_state'] = self.seed
        model.model.set_params(random_state=self.seed)
        model.train(X_rf[train_mask], y[train_mask])

        val_pred = model.predict(X_rf[val_mask])
        cal_pred = model.predict(X_rf[cal_mask])
        test_pred = model.predict(X_rf[test_mask])

        return {
            'model': model,
            'metrics': [
                ModelEvaluator.evaluate_model(y[val_mask], val_pred, 'Random Forest (val)'),
                ModelEvaluator.evaluate_model(y[cal_mask], cal_pred, 'Random Forest (cal)'),
                ModelEvaluator.evaluate_model(y[test_mask], test_pred, 'Random Forest (test)'),
            ],
            'val_pred': val_pred,
            'cal_pred': cal_pred,
            'test_pred': test_pred,
            'val_true': y[val_mask],
            'cal_true': y[cal_mask],
            'test_true': y[test_mask],
            'val_meta': meta[val_mask].reset_index(drop=True),
            'cal_meta': meta[cal_mask].reset_index(drop=True),
            'test_meta': meta[test_mask].reset_index(drop=True),
        }

    def train_parent_model(self, model_name: str, use_transfer: bool) -> dict:
        set_global_seed(self.seed)
        src = self.results['source_sequences']
        tgt = self.results['target_sequences']

        train_mask = self._target_train_mask(tgt['meta'])
        val_mask = self._mask(tgt['meta'], 'val')
        cal_mask = self._mask(tgt['meta'], 'cal')
        test_mask = self._mask(tgt['meta'], 'test')

        model = ContextAwareTransferBackbone(
            shared_dim=len(DataProcessor.SHARED_TRANSFER_FEATURES),
            context_dim=len(DataProcessor.TARGET_CONTEXT_FEATURES),
            n_sites=len(tgt['site_ids']),
            shared_hidden=self.parent_config['shared_hidden'],
            context_hidden=self.parent_config['context_hidden'],
        ).to(DEVICE)

        source_model = None
        if use_transfer:
            src_selected = src['meta']['cell_id'].isin(self.results['selected_source_cells']).to_numpy()
            src_train = self._mask(src['meta'], 'train') & src_selected
            src_val = self._mask(src['meta'], 'val') & src_selected
            if src_train.sum() == 0 or src_val.sum() == 0:
                raise ValueError('Selected source pool does not contain enough train/val windows for pretraining.')

            source_model = SourcePretrainBackbone(
                shared_dim=len(DataProcessor.SHARED_TRANSFER_FEATURES),
                shared_hidden=self.parent_config['shared_hidden'],
            ).to(DEVICE)
            source_train_loader = self._make_loader(
                [src['X_shared'][src_train]],
                src['y_scaled'][src_train],
                batch_size=256,
                shuffle=True,
            )
            source_val_loader = self._make_loader(
                [src['X_shared'][src_val]],
                src['y_scaled'][src_val],
                batch_size=512,
                shuffle=False,
            )
            self._fit_model(
                source_model,
                source_train_loader,
                source_val_loader,
                lr=float(self.transfer_config['pretrain_lr']),
                epochs=int(self.transfer_config['pretrain_epochs']),
                patience=int(self.transfer_config['pretrain_patience']),
            )
            model.shared_encoder.load_state_dict(deepcopy(source_model.shared_encoder.state_dict()))

        train_loader = self._make_loader(
            [tgt['X_shared'][train_mask], tgt['X_context'][train_mask], tgt['site_idx'][train_mask]],
            tgt['y_scaled'][train_mask],
            batch_size=128,
            shuffle=True,
        )
        val_loader = self._make_loader(
            [tgt['X_shared'][val_mask], tgt['X_context'][val_mask], tgt['site_idx'][val_mask]],
            tgt['y_scaled'][val_mask],
            batch_size=256,
            shuffle=False,
        )
        self._fit_model(
            model,
            train_loader,
            val_loader,
            lr=float(self.parent_config['target_lr']),
            epochs=int(self.parent_config['target_epochs']),
            patience=int(self.parent_config['target_patience']),
        )

        pred_arrays = {}
        for split, mask in [('val', val_mask), ('cal', cal_mask), ('test', test_mask)]:
            pred_arrays[split] = self._predict(
                model,
                [tgt['X_shared'][mask], tgt['X_context'][mask], tgt['site_idx'][mask]],
                y_mean=tgt['y_mean'],
                y_std=tgt['y_std'],
            )

        return {
            'model': model,
            'source_model': source_model,
            'metrics': [
                ModelEvaluator.evaluate_model(tgt['y'][val_mask], pred_arrays['val'], f"{model_name} (val)"),
                ModelEvaluator.evaluate_model(tgt['y'][cal_mask], pred_arrays['cal'], f"{model_name} (cal)"),
                ModelEvaluator.evaluate_model(tgt['y'][test_mask], pred_arrays['test'], f"{model_name} (test)"),
            ],
            'val_pred': pred_arrays['val'],
            'cal_pred': pred_arrays['cal'],
            'test_pred': pred_arrays['test'],
            'val_true': tgt['y'][val_mask],
            'cal_true': tgt['y'][cal_mask],
            'test_true': tgt['y'][test_mask],
            'val_meta': tgt['meta'][val_mask].reset_index(drop=True),
            'cal_meta': tgt['meta'][cal_mask].reset_index(drop=True),
            'test_meta': tgt['meta'][test_mask].reset_index(drop=True),
            'config': {
                'model_family': model_name,
                'use_transfer': use_transfer,
                **self.parent_config,
                **self.transfer_config,
            },
        }

    def run_parent_models(self):
        return {
            self.NO_TL_NAME: self.train_parent_model(self.NO_TL_NAME, use_transfer=False),
            self.ALWAYS_TL_NAME: self.train_parent_model(self.ALWAYS_TL_NAME, use_transfer=True),
        }

    def run_baseline_models(self) -> dict[str, dict]:
        set_global_seed(self.seed)
        src = self.results['source_sequences']
        tgt = self.results['target_sequences']

        baseline_source_cells, _ = self._select_source_cells(self.baseline_config['source_mode'])
        baseline_source_mask = src['meta']['cell_id'].isin(baseline_source_cells).to_numpy()
        src_train = self._mask(src['meta'], 'train') & baseline_source_mask
        src_val = self._mask(src['meta'], 'val') & baseline_source_mask
        train_mask = self._target_train_mask(tgt['meta'])
        val_mask = self._mask(tgt['meta'], 'val')
        cal_mask = self._mask(tgt['meta'], 'cal')
        test_mask = self._mask(tgt['meta'], 'test')

        if src_train.sum() == 0 or src_val.sum() == 0:
            raise ValueError('Selected baseline source pool does not contain enough train/val windows for pretraining.')

        baseline_no_tl = LSTMModel(
            sequence_length=self.sequence_length,
            batch_size=self.baseline_config['batch_size'],
            epochs=self.baseline_config['target_epochs'],
            patience=self.baseline_config['target_patience'],
            lr=self.baseline_config['target_lr'],
            hidden_dims=self.baseline_config['hidden_dims'],
            dropout=self.baseline_config['dropout'],
            weight_decay=self.baseline_config['weight_decay'],
            scheduler_factor=self.baseline_config['target_scheduler_factor'],
            scheduler_patience=self.baseline_config['target_scheduler_patience'],
        )
        baseline_no_tl.train(
            tgt['X_shared'][train_mask],
            tgt['y_scaled'][train_mask],
            tgt['X_shared'][val_mask],
            tgt['y_scaled'][val_mask],
        )

        baseline_tl = LSTMTransferLearningModel(
            sequence_length=self.sequence_length,
            batch_size=self.baseline_config['batch_size'],
            epochs=self.baseline_config['tl_epochs'],
            patience=self.baseline_config['tl_patience'],
            lr=self.baseline_config['tl_lr'],
            hidden_dims=self.baseline_config['hidden_dims'],
            dropout=self.baseline_config['dropout'],
            weight_decay=self.baseline_config['weight_decay'],
            scheduler_factor=self.baseline_config['tl_scheduler_factor'],
            scheduler_patience=self.baseline_config['tl_scheduler_patience'],
            n_frozen_layers=self.baseline_config['n_frozen_layers'],
            pretrain_lr=self.baseline_config['pretrain_lr'],
            pretrain_epochs=self.baseline_config['pretrain_epochs'],
            pretrain_patience=self.baseline_config['pretrain_patience'],
            pretrain_scheduler_factor=self.baseline_config['pretrain_scheduler_factor'],
            pretrain_scheduler_patience=self.baseline_config['pretrain_scheduler_patience'],
        )
        baseline_tl.pretrain(
            src['X_shared'][src_train],
            src['y_scaled'][src_train],
            src['X_shared'][src_val],
            src['y_scaled'][src_val],
        )
        baseline_tl.finetune(
            tgt['X_shared'][train_mask],
            tgt['y_scaled'][train_mask],
            tgt['X_shared'][val_mask],
            tgt['y_scaled'][val_mask],
        )

        def build_payload(model_wrapper, model_name: str) -> dict:
            pred_arrays = {}
            for split, mask in [('val', val_mask), ('cal', cal_mask), ('test', test_mask)]:
                pred_arrays[split] = self._predict(
                    model_wrapper.model,
                    [tgt['X_shared'][mask]],
                    y_mean=tgt['y_mean'],
                    y_std=tgt['y_std'],
                )
            return {
                'model': model_wrapper.model,
                'metrics': [
                    ModelEvaluator.evaluate_model(tgt['y'][val_mask], pred_arrays['val'], f"{model_name} (val)"),
                    ModelEvaluator.evaluate_model(tgt['y'][cal_mask], pred_arrays['cal'], f"{model_name} (cal)"),
                    ModelEvaluator.evaluate_model(tgt['y'][test_mask], pred_arrays['test'], f"{model_name} (test)"),
                ],
                'val_pred': pred_arrays['val'],
                'cal_pred': pred_arrays['cal'],
                'test_pred': pred_arrays['test'],
                'val_true': tgt['y'][val_mask],
                'cal_true': tgt['y'][cal_mask],
                'test_true': tgt['y'][test_mask],
                'val_meta': tgt['meta'][val_mask].reset_index(drop=True),
                'cal_meta': tgt['meta'][cal_mask].reset_index(drop=True),
                'test_meta': tgt['meta'][test_mask].reset_index(drop=True),
                'config': {'model_family': model_name, **self.baseline_config},
            }

        return {
            self.BASELINE_NO_TL_NAME: build_payload(baseline_no_tl, self.BASELINE_NO_TL_NAME),
            self.BASELINE_TL_NAME: build_payload(baseline_tl, self.BASELINE_TL_NAME),
        }

    def _window_feature_frame(self) -> pd.DataFrame:
        cached = self.results.get('selector_window_features')
        if cached is not None:
            return cached

        target_sequences = self.results['target_sequences']
        meta = target_sequences['meta'][['site_id', 'target_month', 'split']].copy().reset_index(drop=True)
        meta['target_month'] = pd.to_datetime(meta['target_month'])

        shared_cols = [f'shared_last_{col}' for col in DataProcessor.SHARED_TRANSFER_FEATURES]
        context_cols = [f'context_last_{col}' for col in DataProcessor.TARGET_CONTEXT_FEATURES]

        shared_last = pd.DataFrame(target_sequences['X_shared'][:, -1, :], columns=shared_cols)
        context_last = pd.DataFrame(target_sequences['X_context'][:, -1, :], columns=context_cols)
        out = pd.concat([meta, shared_last, context_last], axis=1)
        self.results['selector_window_features'] = out
        return out

    def _selector_expert_names(self, include_random_forest: bool = False) -> list[str]:
        names = [self.NO_TL_NAME, self.ALWAYS_TL_NAME]
        if include_random_forest:
            names.append('Random Forest')
        return names

    def _selector_expert_payloads(self, include_random_forest: bool = False) -> dict[str, dict]:
        payloads = {
            self.NO_TL_NAME: self.results['parent_payloads'][self.NO_TL_NAME],
            self.ALWAYS_TL_NAME: self.results['parent_payloads'][self.ALWAYS_TL_NAME],
        }
        if include_random_forest:
            rf_payload = self.results.get('random_forest')
            if rf_payload is None:
                rf_payload = self.run_random_forest()
                self.results['random_forest'] = rf_payload
            payloads['Random Forest'] = rf_payload
        return payloads

    def _pred_col(self, expert_name: str) -> str:
        mapping = {
            self.NO_TL_NAME: 'pred_no_tl',
            self.ALWAYS_TL_NAME: 'pred_always_tl',
            'Random Forest': 'pred_random_forest',
        }
        return mapping[expert_name]

    def _build_gate_split_frame(
        self,
        split: str,
        selector_static: pd.DataFrame,
        include_random_forest: bool = False,
    ) -> pd.DataFrame:
        expert_payloads = self._selector_expert_payloads(include_random_forest=include_random_forest)
        anchor_payload = expert_payloads[self.NO_TL_NAME]
        expert_names = self._selector_expert_names(include_random_forest=include_random_forest)
        pred_cols = [self._pred_col(name) for name in expert_names]

        frame = anchor_payload[f'{split}_meta'][['site_id', 'target_month']].copy().reset_index(drop=True)
        frame['target_month'] = pd.to_datetime(frame['target_month'])
        frame['y_true'] = np.asarray(anchor_payload[f'{split}_true']).flatten()
        for expert_name in expert_names:
            frame[self._pred_col(expert_name)] = np.asarray(expert_payloads[expert_name][f'{split}_pred']).flatten()

        frame['month_num'] = frame['target_month'].dt.month.astype(np.int32)
        frame['month_sin'] = np.sin(2 * np.pi * frame['month_num'] / 12.0)
        frame['month_cos'] = np.cos(2 * np.pi * frame['month_num'] / 12.0)

        pred_values = frame[pred_cols].to_numpy(dtype=np.float32)
        frame['pred_min'] = pred_values.min(axis=1)
        frame['pred_max'] = pred_values.max(axis=1)
        frame['pred_std'] = pred_values.std(axis=1)
        frame['pred_mean'] = pred_values.mean(axis=1)
        frame['pred_range'] = frame['pred_always_tl'] - frame['pred_no_tl']
        frame['gap_no_vs_tl'] = frame['pred_range']
        if include_random_forest:
            frame['gap_no_vs_rf'] = frame[self._pred_col('Random Forest')] - frame[self._pred_col(self.NO_TL_NAME)]
            frame['gap_tl_vs_rf'] = frame[self._pred_col('Random Forest')] - frame[self._pred_col(self.ALWAYS_TL_NAME)]

        frame = frame.merge(selector_static, on='site_id', how='left')
        frame = frame.merge(
            self._window_feature_frame().drop(columns='split'),
            on=['site_id', 'target_month'],
            how='left',
        )
        return frame

    @staticmethod
    def _mean_site_rmse(y_true: np.ndarray, y_pred: np.ndarray, meta: pd.DataFrame) -> float:
        site_rmse = LSTMTransferLearningImplementation._site_level_rmse(y_true, y_pred, meta, 'rmse')
        return float(site_rmse['rmse'].mean())

    @staticmethod
    def _ordered_unique(items: list[str]) -> list[str]:
        return list(dict.fromkeys(items))

    def _selector_feature_groups(self) -> dict[str, list[str]]:
        shared_cols = [f'shared_last_{col}' for col in DataProcessor.SHARED_TRANSFER_FEATURES]
        context_cols = [f'context_last_{col}' for col in DataProcessor.TARGET_CONTEXT_FEATURES]
        expert_cols = [
            'pred_no_tl', 'pred_always_tl',
            'pred_min', 'pred_max', 'pred_range', 'pred_std', 'pred_mean',
            'gap_no_vs_tl',
        ]
        calendar_cols = ['month_sin', 'month_cos']
        similarity_cols = [
            'cosine_similarity', 'months_used',
            'sim_mean', 'sim_std', 'sim_max', 'sim_min', 'sim_q90', 'sim_q95',
            'sim_top3_mean', 'sim_top3_std', 'sim_top5_mean', 'sim_top5_std', 'sim_top10_mean', 'sim_top10_std',
        ]
        site_cols = [
            'train_rows', 'wtda_mean', 'wtda_std', 'pr_a_mean', 'pr_a_std', 'sm_a_mean', 'sm_a_std',
            'tsmp_wtda_mean', 'tsmp_wtda_std', 'pumping_log_mean', 'pumping_log_std', 'lon_norm', 'lat_norm',
        ]
        window_state_cols = shared_cols + context_cols

        return {
            'expert': expert_cols,
            'calendar': calendar_cols,
            'similarity': similarity_cols,
            'site': site_cols,
            'window_state': window_state_cols,
            'full': self._ordered_unique(expert_cols + calendar_cols + similarity_cols + site_cols + window_state_cols),
        }

    def _selector_feature_specs(self) -> dict[str, dict[str, object]]:
        feature_groups = self._selector_feature_groups()
        return {
            self.SELECTOR_MAIN_KEY: {
                'label': 'Main gate (context + similarity + target state)',
                'feature_cols': self._ordered_unique(
                    feature_groups['calendar'] + feature_groups['similarity'] + feature_groups['site'] + feature_groups['window_state']
                ),
            },
            self.SELECTOR_FULL_KEY: {
                'label': 'Full gate (+ expert disagreement)',
                'feature_cols': feature_groups['full'],
            },
            'no_similarity': {
                'label': 'No similarity diagnostics',
                'feature_cols': self._ordered_unique(
                    feature_groups['calendar'] + feature_groups['site'] + feature_groups['window_state']
                ),
            },
            'no_window_state': {
                'label': 'No target state',
                'feature_cols': self._ordered_unique(
                    feature_groups['calendar'] + feature_groups['similarity'] + feature_groups['site']
                ),
            },
        }

    def _selector_candidate_factories(
        self,
        feature_dim: int,
        include_neural_gate: bool = False,
    ) -> dict[str, object]:
        from sklearn.ensemble import HistGradientBoostingRegressor
        from sklearn.linear_model import Ridge

        factories = {
            'ridge_env': lambda: Ridge(alpha=10.0, random_state=self.seed),
            'rf_env': lambda: RandomForestRegressor(
                n_estimators=700,
                max_depth=10,
                min_samples_leaf=8,
                random_state=self.seed,
                n_jobs=-1,
            ),
            'hgb_env': lambda: HistGradientBoostingRegressor(
                max_depth=5,
                max_iter=500,
                learning_rate=0.03,
                min_samples_leaf=20,
                random_state=self.seed,
            ),
        }
        if include_neural_gate:
            factories['mlp_env'] = lambda: NeuralGateRegressor(
                input_dim=feature_dim,
                seed=self.seed,
                hidden_dims=self.selector_config['neural_hidden_dims'],
                dropout=self.selector_config['neural_dropout'],
                lr=self.selector_config['neural_lr'],
                epochs=self.selector_config['neural_epochs'],
                patience=self.selector_config['neural_patience'],
                batch_size=self.selector_config['neural_batch_size'],
                weight_decay=self.selector_config['neural_weight_decay'],
            )
        return factories

    def _apply_transfer_gate(self, gate_frames: dict[str, pd.DataFrame], selector_by_site: pd.DataFrame, model_label: str) -> dict:
        split_payloads = {}
        for split, frame in gate_frames.items():
            routed = frame.copy().reset_index(drop=True)
            routed['gate_gamma'] = routed['gate_gamma'].clip(0.0, 1.0)
            routed['selected_expert'] = np.where(
                routed['gate_gamma'] >= 0.5,
                self.ALWAYS_TL_NAME,
                self.NO_TL_NAME,
            )
            routed['selected_parent'] = routed['selected_expert']
            routed['use_transfer'] = routed['selected_expert'].eq(self.ALWAYS_TL_NAME)

            gamma = routed['gate_gamma'].to_numpy(dtype=np.float32)
            pred_no = routed['pred_no_tl'].to_numpy(dtype=np.float32)
            pred_tl = routed['pred_always_tl'].to_numpy(dtype=np.float32)
            pred = ((1.0 - gamma) * pred_no + gamma * pred_tl).reshape(-1, 1)
            true = routed['y_true'].to_numpy(dtype=np.float32).reshape(-1, 1)

            split_payloads[split] = {
                'pred': pred,
                'true': true,
                'meta': routed,
            }

        val_pred = split_payloads['val']['pred']
        val_true = split_payloads['val']['true']
        val_meta = split_payloads['val']['meta']
        cal_pred = split_payloads['cal']['pred']
        cal_true = split_payloads['cal']['true']
        cal_meta = split_payloads['cal']['meta']
        test_pred = split_payloads['test']['pred']
        test_true = split_payloads['test']['true']
        test_meta = split_payloads['test']['meta']

        return {
            'metrics': [
                ModelEvaluator.evaluate_model(val_true, val_pred, f'{model_label} (val)'),
                ModelEvaluator.evaluate_model(cal_true, cal_pred, f'{model_label} (cal)'),
                ModelEvaluator.evaluate_model(test_true, test_pred, f'{model_label} (test)'),
            ],
            'val_pred': val_pred,
            'cal_pred': cal_pred,
            'test_pred': test_pred,
            'val_true': val_true,
            'cal_true': cal_true,
            'test_true': test_true,
            'val_meta': val_meta.reset_index(drop=True),
            'cal_meta': cal_meta.reset_index(drop=True),
            'test_meta': test_meta.reset_index(drop=True),
            'selector_by_site': selector_by_site.reset_index(drop=True),
            'selector_allocation': DomainAnalyzer.selector_allocation_summary(test_meta),
        }

    def _apply_site_policy(self, selector_by_site: pd.DataFrame, model_label: str, include_random_forest: bool = False) -> dict:
        expert_payloads = self._selector_expert_payloads(include_random_forest=include_random_forest)
        expert_names = self._selector_expert_names(include_random_forest=include_random_forest)

        def route_split(split: str):
            split_meta = expert_payloads[self.NO_TL_NAME][f'{split}_meta'].copy().reset_index(drop=True)
            routed = split_meta.merge(
                selector_by_site[['site_id', 'selected_parent', 'selected_expert', 'use_transfer', 'cosine_similarity', 'months_used']],
                on='site_id',
                how='left',
            )
            routed['selected_expert'] = routed['selected_expert'].fillna(routed['selected_parent']).fillna(self.NO_TL_NAME)
            routed['selected_parent'] = routed['selected_expert']

            pred = np.asarray(expert_payloads[self.NO_TL_NAME][f'{split}_pred']).flatten()
            for expert_name in expert_names[1:]:
                expert_pred = np.asarray(expert_payloads[expert_name][f'{split}_pred']).flatten()
                pred = np.where(routed['selected_expert'].eq(expert_name).to_numpy(), expert_pred, pred)
            return pred.reshape(-1, 1), expert_payloads[self.NO_TL_NAME][f'{split}_true'], routed

        val_pred, val_true, val_meta = route_split('val')
        cal_pred, cal_true, cal_meta = route_split('cal')
        test_pred, test_true, test_meta = route_split('test')

        return {
            'metrics': [
                ModelEvaluator.evaluate_model(val_true, val_pred, f'{model_label} (val)'),
                ModelEvaluator.evaluate_model(cal_true, cal_pred, f'{model_label} (cal)'),
                ModelEvaluator.evaluate_model(test_true, test_pred, f'{model_label} (test)'),
            ],
            'val_pred': val_pred,
            'cal_pred': cal_pred,
            'test_pred': test_pred,
            'val_true': val_true,
            'cal_true': cal_true,
            'test_true': test_true,
            'val_meta': val_meta.reset_index(drop=True),
            'cal_meta': cal_meta.reset_index(drop=True),
            'test_meta': test_meta.reset_index(drop=True),
            'selector_by_site': selector_by_site.reset_index(drop=True),
            'selector_allocation': DomainAnalyzer.selector_allocation_summary(test_meta),
        }

    def _fit_selector_payload(
        self,
        selector_static: pd.DataFrame,
        val_frame: pd.DataFrame,
        cal_frame: pd.DataFrame,
        test_frame: pd.DataFrame,
        feature_cols: list[str],
        model_label: str,
        compute_feature_importance: bool = True,
        include_neural_gate: bool = False,
        candidate_subset: list[str] | None = None,
    ) -> dict:
        from sklearn.inspection import permutation_importance
        from sklearn.model_selection import GroupKFold

        median_map = cal_frame[feature_cols].median(numeric_only=True)
        X_cal = cal_frame[feature_cols].fillna(median_map).to_numpy(dtype=np.float32)
        y_cal = cal_frame['y_true'].to_numpy(dtype=np.float32)
        pred_no_cal = cal_frame['pred_no_tl'].to_numpy(dtype=np.float32)
        pred_tl_cal = cal_frame['pred_always_tl'].to_numpy(dtype=np.float32)
        range_cal = pred_tl_cal - pred_no_cal
        selector_groups = cal_frame['site_id'].to_numpy()

        range_safe = np.where(np.abs(range_cal) < 1e-6, 1.0, range_cal)
        gamma_star = np.clip((y_cal - pred_no_cal) / range_safe, 0.0, 1.0).astype(np.float32)
        selector_weights = np.square(range_cal) + 0.05

        candidate_factories = self._selector_candidate_factories(
            feature_dim=int(len(feature_cols)),
            include_neural_gate=include_neural_gate,
        )
        if candidate_subset is not None:
            candidate_factories = {name: candidate_factories[name] for name in candidate_subset}

        candidate_rows = []
        candidate_oof = {}
        group_kfold = GroupKFold(n_splits=5)
        for candidate_name, factory in candidate_factories.items():
            oof_gamma = np.full(len(cal_frame), 0.5, dtype=np.float32)
            for train_idx, hold_idx in group_kfold.split(X_cal, gamma_star, selector_groups):
                fold_model = factory()
                fold_model.fit(X_cal[train_idx], gamma_star[train_idx], sample_weight=selector_weights[train_idx])
                oof_gamma[hold_idx] = np.clip(fold_model.predict(X_cal[hold_idx]), 0.0, 1.0)

            oof_pred = (1.0 - oof_gamma) * pred_no_cal + oof_gamma * pred_tl_cal
            candidate_oof[candidate_name] = oof_gamma
            candidate_rows.append(
                {
                    'candidate_model': candidate_name,
                    'rmse_oof': float(np.sqrt(mean_squared_error(y_cal, oof_pred))),
                    'mean_site_rmse_oof': self._mean_site_rmse(y_cal, oof_pred, cal_frame),
                    'share_transfer_oof': float((oof_gamma >= 0.5).mean()),
                    'share_upper_envelope_oof': float((oof_gamma >= 0.5).mean()),
                    'mean_gate_gamma_oof': float(oof_gamma.mean()),
                }
            )

        selector_search = (
            pd.DataFrame(candidate_rows)
            .sort_values(['mean_site_rmse_oof', 'rmse_oof', 'candidate_model'])
            .reset_index(drop=True)
        )
        best_candidate = str(selector_search.iloc[0]['candidate_model'])
        selector_model = candidate_factories[best_candidate]()
        selector_model.fit(X_cal, gamma_star, sample_weight=selector_weights)

        gate_frames = {}
        for split_name, split_frame in [('val', val_frame), ('cal', cal_frame), ('test', test_frame)]:
            split_X = split_frame[feature_cols].fillna(median_map).to_numpy(dtype=np.float32)
            split_frame = split_frame.copy()
            split_frame['gate_gamma'] = np.clip(selector_model.predict(split_X), 0.0, 1.0).astype(np.float32)
            if split_name == self.selector_split:
                split_frame['gate_gamma_oof'] = candidate_oof[best_candidate]
            gate_frames[split_name] = split_frame

        cal_gamma_fit = gate_frames['cal']['gate_gamma'].to_numpy(dtype=np.float32)
        cal_pred_fit = (1.0 - cal_gamma_fit) * pred_no_cal + cal_gamma_fit * pred_tl_cal
        cal_pred_oof = (1.0 - candidate_oof[best_candidate]) * pred_no_cal + candidate_oof[best_candidate] * pred_tl_cal

        selector_by_site = selector_static.copy()
        selector_by_site = selector_by_site.merge(
            self._site_level_rmse(
                self.results['parent_payloads'][self.NO_TL_NAME][f'{self.selector_split}_true'],
                self.results['parent_payloads'][self.NO_TL_NAME][f'{self.selector_split}_pred'],
                self.results['parent_payloads'][self.NO_TL_NAME][f'{self.selector_split}_meta'],
                'selector_rmse_no_tl',
            ),
            on='site_id',
            how='inner',
        )
        selector_by_site = selector_by_site.merge(
            self._site_level_rmse(
                self.results['parent_payloads'][self.ALWAYS_TL_NAME][f'{self.selector_split}_true'],
                self.results['parent_payloads'][self.ALWAYS_TL_NAME][f'{self.selector_split}_pred'],
                self.results['parent_payloads'][self.ALWAYS_TL_NAME][f'{self.selector_split}_meta'],
                'selector_rmse_always_tl',
            ),
            on='site_id',
            how='inner',
        )
        selector_by_site = selector_by_site.merge(
            self._site_level_rmse(
                gate_frames['cal']['y_true'].to_numpy(dtype=np.float32).reshape(-1, 1),
                cal_pred_fit.reshape(-1, 1),
                gate_frames['cal'],
                'selector_rmse_selective_learning',
            ),
            on='site_id',
            how='inner',
        )
        selector_by_site = selector_by_site.merge(
            self._site_level_rmse(
                gate_frames['cal']['y_true'].to_numpy(dtype=np.float32).reshape(-1, 1),
                cal_pred_oof.reshape(-1, 1),
                gate_frames['cal'],
                'selector_rmse_selective_learning_oof',
            ),
            on='site_id',
            how='inner',
        )
        selector_by_site = selector_by_site.merge(
            self._site_level_counts(gate_frames['cal'], 'selector_observations'),
            on='site_id',
            how='left',
        )

        gamma_summary = (
            gate_frames['cal']
            .groupby('site_id', as_index=False)
            .agg(
                selector_gamma_mean=('gate_gamma', 'mean'),
                selector_gamma_std=('gate_gamma', 'std'),
                selector_gamma_oof_mean=('gate_gamma_oof', 'mean'),
                selector_gamma_oof_std=('gate_gamma_oof', 'std'),
                transfer_share=('gate_gamma', lambda s: float((np.asarray(s) >= 0.5).mean())),
                transfer_share_oof=('gate_gamma_oof', lambda s: float((np.asarray(s) >= 0.5).mean())),
            )
        )
        gamma_summary['upper_envelope_share'] = gamma_summary['transfer_share']
        gamma_summary['upper_envelope_share_oof'] = gamma_summary['transfer_share_oof']

        selected_expert_frame = gate_frames['cal'][['site_id', 'gate_gamma']].copy()
        selected_expert_frame['selected_expert'] = np.where(
            selected_expert_frame['gate_gamma'] >= 0.5,
            self.ALWAYS_TL_NAME,
            self.NO_TL_NAME,
        )
        dominant_expert = (
            selected_expert_frame.groupby(['site_id', 'selected_expert'], as_index=False)
            .size()
            .sort_values(['site_id', 'size', 'selected_expert'], ascending=[True, False, True])
            .drop_duplicates('site_id')
            .rename(columns={'size': 'selected_expert_count'})
        )

        selector_by_site = selector_by_site.merge(gamma_summary, on='site_id', how='left')
        selector_by_site = selector_by_site.merge(dominant_expert, on='site_id', how='left')
        selector_by_site[['selector_gamma_std', 'selector_gamma_oof_std']] = (
            selector_by_site[['selector_gamma_std', 'selector_gamma_oof_std']].fillna(0.0)
        )
        selector_by_site['selector_model'] = best_candidate
        selector_by_site['selected_parent'] = selector_by_site['selected_expert'].fillna(self.NO_TL_NAME)
        selector_by_site['use_transfer'] = selector_by_site['selected_parent'].eq(self.ALWAYS_TL_NAME)
        selector_by_site['selected_selector_rmse'] = selector_by_site['selector_rmse_selective_learning']
        selector_by_site['selector_gain_vs_no_tl'] = (
            selector_by_site['selector_rmse_selective_learning'] - selector_by_site['selector_rmse_no_tl']
        )
        selector_by_site['selector_gain_vs_no_tl_oof'] = (
            selector_by_site['selector_rmse_selective_learning_oof'] - selector_by_site['selector_rmse_no_tl']
        )

        if compute_feature_importance:
            feature_importance_raw = permutation_importance(
                selector_model,
                X_cal,
                gamma_star,
                n_repeats=8,
                random_state=self.seed,
                scoring='neg_mean_squared_error',
            )
            selector_feature_importance = (
                pd.DataFrame(
                    {
                        'feature': feature_cols,
                        'importance': feature_importance_raw.importances_mean,
                        'importance_std': feature_importance_raw.importances_std,
                    }
                )
                .sort_values('importance', ascending=False)
                .reset_index(drop=True)
            )
        else:
            selector_feature_importance = pd.DataFrame(columns=['feature', 'importance', 'importance_std'])

        payload = self._apply_transfer_gate(gate_frames, selector_by_site, model_label)
        payload['selector_search'] = selector_search
        payload['best_candidate'] = best_candidate
        payload['selector_feature_importance'] = selector_feature_importance
        payload['selector_feature_cols'] = feature_cols
        payload['selector_feature_set'] = model_label
        return payload

    def run_similarity_selector(self, compute_feature_importance: bool = True) -> dict:
        selector_static = (
            self.results['best_match_lookup'][['site_id', 'cell_id', 'cosine_similarity', 'months_used']]
            .drop_duplicates('site_id')
            .merge(self.build_selector_features(), on='site_id', how='left')
        )

        cal_frame = self._build_gate_split_frame(self.selector_split, selector_static)
        val_frame = self._build_gate_split_frame('val', selector_static)
        test_frame = self._build_gate_split_frame('test', selector_static)
        main_spec = self._selector_feature_specs()[self.SELECTOR_MAIN_KEY]
        return self._fit_selector_payload(
            selector_static=selector_static,
            val_frame=val_frame,
            cal_frame=cal_frame,
            test_frame=test_frame,
            feature_cols=main_spec['feature_cols'],
            model_label=self.SELECTIVE_TL_NAME,
            compute_feature_importance=compute_feature_importance,
        )

    def run_site_hard_selector(self) -> dict:
        expert_payloads = self._selector_expert_payloads(include_random_forest=False)
        expert_names = self._selector_expert_names(include_random_forest=False)
        selector_split = self.selector_split

        policy_by_site = self.results['best_match_lookup'][['site_id', 'cell_id', 'cosine_similarity', 'months_used']].drop_duplicates('site_id').copy()
        rmse_cols = []
        for expert_name in expert_names:
            rmse_col = f"selector_rmse_{self._pred_col(expert_name).replace('pred_', '')}"
            policy_by_site = policy_by_site.merge(
                self._site_level_rmse(
                    expert_payloads[expert_name][f'{selector_split}_true'],
                    expert_payloads[expert_name][f'{selector_split}_pred'],
                    expert_payloads[expert_name][f'{selector_split}_meta'],
                    rmse_col,
                ),
                on='site_id',
                how='inner',
            )
            rmse_cols.append(rmse_col)

        no_col = f"selector_rmse_{self._pred_col(self.NO_TL_NAME).replace('pred_', '')}"
        tl_col = f"selector_rmse_{self._pred_col(self.ALWAYS_TL_NAME).replace('pred_', '')}"
        policy_by_site['calibration_margin_no_tl_vs_always_tl'] = policy_by_site[tl_col] - policy_by_site[no_col]

        # Conservative site-hard policy: fixed transfer is the default parent, and the selector
        # switches to the local parent only when the calibration advantage is large enough and
        # the source-target overlap is long enough to make that site-level decision stable.
        use_no_tl = (
            (policy_by_site['calibration_margin_no_tl_vs_always_tl'] > self.SITE_SELECTOR_MIN_NO_TL_MARGIN)
            & (policy_by_site['months_used'].fillna(0) >= self.SITE_SELECTOR_MIN_OVERLAP_MONTHS)
        )
        policy_by_site['selected_expert'] = np.where(use_no_tl, self.NO_TL_NAME, self.ALWAYS_TL_NAME)
        policy_by_site['selected_parent'] = policy_by_site['selected_expert']
        policy_by_site['use_transfer'] = policy_by_site['selected_expert'].eq(self.ALWAYS_TL_NAME)
        policy_by_site['selected_selector_rmse'] = np.where(use_no_tl, policy_by_site[no_col], policy_by_site[tl_col])
        policy_by_site['selector_policy'] = 'cal_margin_gt_0.02_and_overlap_ge_72_default_always_tl'
        policy_by_site['selector_margin_threshold'] = self.SITE_SELECTOR_MIN_NO_TL_MARGIN
        policy_by_site['selector_overlap_threshold_months'] = self.SITE_SELECTOR_MIN_OVERLAP_MONTHS

        return self._apply_site_policy(policy_by_site, self.SITE_HARD_TL_NAME)

    def build_site_transfer_effects(self, site_metric_wide: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
        site_effects = site_metric_wide.merge(self.results['best_match_lookup'], on='site_id', how='left')
        site_effects['delta_rmse_always_tl_vs_no_tl'] = site_effects[self.ALWAYS_TL_NAME] - site_effects[self.NO_TL_NAME]
        site_effects['delta_rmse_selective_tl_vs_no_tl'] = site_effects[self.SELECTIVE_TL_NAME] - site_effects[self.NO_TL_NAME]
        site_effects['delta_rmse_site_hard_tl_vs_no_tl'] = site_effects[self.SITE_HARD_TL_NAME] - site_effects[self.NO_TL_NAME]
        site_effects['helped_by_always_tl'] = site_effects['delta_rmse_always_tl_vs_no_tl'] < 0
        site_effects['helped_by_selective_tl'] = site_effects['delta_rmse_selective_tl_vs_no_tl'] < 0

        summary_rows = []
        for strategy, delta_col in [
            ('always_tl', 'delta_rmse_always_tl_vs_no_tl'),
            ('selective_learning', 'delta_rmse_selective_tl_vs_no_tl'),
            ('site_hard_selector', 'delta_rmse_site_hard_tl_vs_no_tl'),
        ]:
            valid = site_effects[['cosine_similarity', delta_col]].dropna()
            summary_rows.append(
                {
                    'strategy': strategy,
                    'n_sites': int(len(valid)),
                    'mean_delta_rmse': float(valid[delta_col].mean()),
                    'median_delta_rmse': float(valid[delta_col].median()),
                    'helped_wells': int((valid[delta_col] < 0).sum()),
                    'harmed_wells': int((valid[delta_col] > 0).sum()),
                    'spearman_similarity_vs_delta': float(valid['cosine_similarity'].corr(valid[delta_col], method='spearman')),
                }
            )

        return site_effects, pd.DataFrame(summary_rows)

    def run_selector_ablations(self) -> pd.DataFrame:
        selector_static = (
            self.results['best_match_lookup'][['site_id', 'cell_id', 'cosine_similarity', 'months_used']]
            .drop_duplicates('site_id')
            .merge(self.build_selector_features(), on='site_id', how='left')
        )
        cal_frame = self._build_gate_split_frame(self.selector_split, selector_static)
        val_frame = self._build_gate_split_frame('val', selector_static)
        test_frame = self._build_gate_split_frame('test', selector_static)
        feature_specs = self._selector_feature_specs()
        ablation_specs = [
            {
                'ablation_key': self.SELECTOR_MAIN_KEY,
                'ablation_label': feature_specs[self.SELECTOR_MAIN_KEY]['label'],
                'feature_cols': feature_specs[self.SELECTOR_MAIN_KEY]['feature_cols'],
                'include_neural_gate': False,
                'candidate_subset': None,
            },
            {
                'ablation_key': 'deep_neural_gate',
                'ablation_label': 'Deep neural gate',
                'feature_cols': feature_specs[self.SELECTOR_MAIN_KEY]['feature_cols'],
                'include_neural_gate': True,
                'candidate_subset': ['mlp_env'],
            },
            {
                'ablation_key': self.SELECTOR_FULL_KEY,
                'ablation_label': feature_specs[self.SELECTOR_FULL_KEY]['label'],
                'feature_cols': feature_specs[self.SELECTOR_FULL_KEY]['feature_cols'],
                'include_neural_gate': False,
                'candidate_subset': None,
            },
            {
                'ablation_key': 'no_similarity',
                'ablation_label': feature_specs['no_similarity']['label'],
                'feature_cols': feature_specs['no_similarity']['feature_cols'],
                'include_neural_gate': False,
                'candidate_subset': None,
            },
            {
                'ablation_key': 'no_window_state',
                'ablation_label': feature_specs['no_window_state']['label'],
                'feature_cols': feature_specs['no_window_state']['feature_cols'],
                'include_neural_gate': False,
                'candidate_subset': None,
            },
        ]

        rows = []
        main_rmse = None
        for spec in ablation_specs:
            ablation_key = spec['ablation_key']
            ablation_label = spec['ablation_label']
            feature_cols = spec['feature_cols']
            if ablation_key == self.SELECTOR_MAIN_KEY and 'selective_tl' in self.results:
                payload = self.results['selective_tl']
                best_candidate = str(payload.get('best_candidate', 'reused_main_selector'))
            else:
                payload = self._fit_selector_payload(
                    selector_static=selector_static,
                    val_frame=val_frame,
                    cal_frame=cal_frame,
                    test_frame=test_frame,
                    feature_cols=feature_cols,
                    model_label=f'{self.SELECTIVE_TL_NAME}: {ablation_label}',
                    compute_feature_importance=False,
                    include_neural_gate=bool(spec['include_neural_gate']),
                    candidate_subset=spec['candidate_subset'],
                )
                best_candidate = str(payload.get('best_candidate', 'unknown'))

            test_metric = next(row for row in payload['metrics'] if str(row['model']).endswith('(test)'))
            cal_metric = next(row for row in payload['metrics'] if str(row['model']).endswith('(cal)'))
            if ablation_key == self.SELECTOR_MAIN_KEY:
                main_rmse = float(test_metric['rmse'])

            rows.append(
                {
                    'ablation_key': ablation_key,
                    'ablation_label': ablation_label,
                    'selector_model': best_candidate,
                    'n_features': int(len(feature_cols)),
                    'rmse_cal': float(cal_metric['rmse']),
                    'rmse_test': float(test_metric['rmse']),
                    'mae_test': float(test_metric['mae']),
                    'r2_test': float(test_metric['r2']),
                    'delta_rmse_vs_always_tl': float(test_metric['rmse'] - self.results['summary_wide'].loc[
                        self.results['summary_wide']['model_family'] == self.ALWAYS_TL_NAME, 'rmse_test'
                    ].iloc[0]),
                    'delta_rmse_vs_no_tl': float(test_metric['rmse'] - self.results['summary_wide'].loc[
                        self.results['summary_wide']['model_family'] == self.NO_TL_NAME, 'rmse_test'
                    ].iloc[0]),
                }
            )

        ablation_df = pd.DataFrame(rows)
        if ablation_df.empty:
            return ablation_df

        if main_rmse is None:
            main_rmse = float(ablation_df.loc[ablation_df['ablation_key'] == self.SELECTOR_MAIN_KEY, 'rmse_test'].iloc[0])
        ablation_df['delta_rmse_vs_main'] = ablation_df['rmse_test'] - float(main_rmse)

        site_hard_rmse = float(
            self.results['summary_wide'].loc[
                self.results['summary_wide']['model_family'] == self.SITE_HARD_TL_NAME,
                'rmse_test',
            ].iloc[0]
        )
        ablation_df = pd.concat(
            [
                ablation_df,
                pd.DataFrame(
                    [
                        {
                            'ablation_key': 'site_hard_selector',
                            'ablation_label': self.SITE_HARD_TL_NAME,
                            'selector_model': 'hard_site_policy',
                            'n_features': 0,
                            'rmse_cal': float(
                                self.results['summary_wide'].loc[
                                    self.results['summary_wide']['model_family'] == self.SITE_HARD_TL_NAME,
                                    'rmse_cal',
                                ].iloc[0]
                            ),
                            'rmse_test': site_hard_rmse,
                            'mae_test': float(
                                self.results['summary_wide'].loc[
                                    self.results['summary_wide']['model_family'] == self.SITE_HARD_TL_NAME,
                                    'mae_test',
                                ].iloc[0]
                            ),
                            'r2_test': float(
                                self.results['summary_wide'].loc[
                                    self.results['summary_wide']['model_family'] == self.SITE_HARD_TL_NAME,
                                    'r2_test',
                                ].iloc[0]
                            ),
                            'delta_rmse_vs_always_tl': site_hard_rmse - float(
                                self.results['summary_wide'].loc[
                                    self.results['summary_wide']['model_family'] == self.ALWAYS_TL_NAME,
                                    'rmse_test',
                                ].iloc[0]
                            ),
                            'delta_rmse_vs_no_tl': site_hard_rmse - float(
                                self.results['summary_wide'].loc[
                                    self.results['summary_wide']['model_family'] == self.NO_TL_NAME,
                                    'rmse_test',
                                ].iloc[0]
                            ),
                            'delta_rmse_vs_main': site_hard_rmse - float(main_rmse),
                        }
                    ]
                ),
            ],
            ignore_index=True,
        )
        return ablation_df.sort_values(['rmse_test', 'ablation_label']).reset_index(drop=True)

    def build_negative_transfer_diagnostics(
        self,
        model_payloads: dict[str, dict],
        site_metric_wide: pd.DataFrame,
        split: str = 'test',
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        required_models = [self.NO_TL_NAME, self.ALWAYS_TL_NAME, self.SELECTIVE_TL_NAME]
        if any(model_name not in model_payloads for model_name in required_models):
            return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

        rename_map = {
            self.NO_TL_NAME: ('pred_no_tl', 'abs_error_no_tl'),
            self.ALWAYS_TL_NAME: ('pred_always_tl', 'abs_error_always_tl'),
            self.SELECTIVE_TL_NAME: ('pred_selective_learning', 'abs_error_selective_learning'),
            'Random Forest': ('pred_random_forest', 'abs_error_random_forest'),
        }

        merged = None
        for model_name, (pred_col, error_col) in rename_map.items():
            if model_name not in model_payloads:
                continue
            frame = (
                self._error_frame(model_payloads[model_name], split)
                .rename(columns={'y_pred': pred_col, 'abs_error': error_col})
            )
            keep_cols = ['site_id', 'target_month', 'y_true', pred_col, error_col]
            merged = frame[keep_cols].copy() if merged is None else merged.merge(
                frame[['site_id', 'target_month', pred_col, error_col]],
                on=['site_id', 'target_month'],
                how='inner',
            )

        if merged is None or merged.empty:
            return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

        merged['delta_abs_error_always_tl_vs_no_tl'] = merged['abs_error_always_tl'] - merged['abs_error_no_tl']
        merged['delta_abs_error_selective_vs_no_tl'] = merged['abs_error_selective_learning'] - merged['abs_error_no_tl']
        merged['delta_abs_error_selective_vs_always_tl'] = merged['abs_error_selective_learning'] - merged['abs_error_always_tl']
        merged['always_tl_observation_regime'] = np.where(
            merged['delta_abs_error_always_tl_vs_no_tl'] < 0,
            'helped',
            np.where(merged['delta_abs_error_always_tl_vs_no_tl'] > 0, 'harmed', 'tie'),
        )
        merged['selective_partial_rescue'] = (
            (merged['delta_abs_error_always_tl_vs_no_tl'] > 0)
            & (merged['delta_abs_error_selective_vs_always_tl'] < 0)
        )
        merged['selective_full_rescue'] = (
            (merged['delta_abs_error_always_tl_vs_no_tl'] > 0)
            & (merged['delta_abs_error_selective_vs_no_tl'] < 0)
        )

        site_summary = (
            merged.groupby('site_id', as_index=False)
            .agg(
                n_observations=('site_id', 'size'),
                share_obs_helped_by_always_tl=('delta_abs_error_always_tl_vs_no_tl', lambda s: float((np.asarray(s) < 0).mean())),
                share_obs_harmed_by_always_tl=('delta_abs_error_always_tl_vs_no_tl', lambda s: float((np.asarray(s) > 0).mean())),
                mean_delta_abs_error_always_tl_vs_no_tl=('delta_abs_error_always_tl_vs_no_tl', 'mean'),
                mean_delta_abs_error_selective_vs_no_tl=('delta_abs_error_selective_vs_no_tl', 'mean'),
                mean_delta_abs_error_selective_vs_always_tl=('delta_abs_error_selective_vs_always_tl', 'mean'),
                selective_partial_rescue_rate=('selective_partial_rescue', 'mean'),
                selective_full_rescue_rate=('selective_full_rescue', 'mean'),
            )
        )
        site_summary = site_summary.merge(
            self.results['best_match_lookup'][['site_id', 'cosine_similarity', 'months_used']],
            on='site_id',
            how='left',
        )
        site_summary = site_summary.merge(
            site_metric_wide[[ 'site_id', self.NO_TL_NAME, self.ALWAYS_TL_NAME, self.SELECTIVE_TL_NAME ]],
            on='site_id',
            how='left',
        )
        site_summary['delta_rmse_always_tl_vs_no_tl'] = site_summary[self.ALWAYS_TL_NAME] - site_summary[self.NO_TL_NAME]
        site_summary['delta_rmse_selective_vs_no_tl'] = site_summary[self.SELECTIVE_TL_NAME] - site_summary[self.NO_TL_NAME]
        site_summary['delta_rmse_selective_vs_always_tl'] = site_summary[self.SELECTIVE_TL_NAME] - site_summary[self.ALWAYS_TL_NAME]
        site_summary['always_tl_site_regime'] = np.where(
            site_summary['delta_rmse_always_tl_vs_no_tl'] < 0,
            'helped',
            np.where(site_summary['delta_rmse_always_tl_vs_no_tl'] > 0, 'harmed', 'tie'),
        )
        site_summary['selective_partial_rescue_site'] = (
            (site_summary['delta_rmse_always_tl_vs_no_tl'] > 0)
            & (site_summary['delta_rmse_selective_vs_always_tl'] < 0)
        )
        site_summary['selective_full_rescue_site'] = (
            (site_summary['delta_rmse_always_tl_vs_no_tl'] > 0)
            & (site_summary['delta_rmse_selective_vs_no_tl'] < 0)
        )

        summary_rows = []
        summary_specs = [
            ('site', 'all_sites', site_summary),
            ('site', 'always_tl_harmed_sites', site_summary[site_summary['always_tl_site_regime'] == 'harmed']),
            ('site', 'always_tl_helped_sites', site_summary[site_summary['always_tl_site_regime'] == 'helped']),
            ('observation', 'all_observations', merged),
            ('observation', 'always_tl_harmed_observations', merged[merged['always_tl_observation_regime'] == 'harmed']),
            ('observation', 'always_tl_helped_observations', merged[merged['always_tl_observation_regime'] == 'helped']),
        ]
        for analysis_unit, subset_label, subset in summary_specs:
            if subset.empty:
                continue
            if analysis_unit == 'site':
                summary_rows.append(
                    {
                        'analysis_unit': analysis_unit,
                        'subset_label': subset_label,
                        'n': int(len(subset)),
                        'mean_always_tl_delta': float(subset['delta_rmse_always_tl_vs_no_tl'].mean()),
                        'mean_selective_delta_vs_no_tl': float(subset['delta_rmse_selective_vs_no_tl'].mean()),
                        'mean_selective_delta_vs_always_tl': float(subset['delta_rmse_selective_vs_always_tl'].mean()),
                        'always_tl_help_rate': float((subset['delta_rmse_always_tl_vs_no_tl'] < 0).mean()),
                        'always_tl_harm_rate': float((subset['delta_rmse_always_tl_vs_no_tl'] > 0).mean()),
                        'selective_partial_rescue_rate': float(subset['selective_partial_rescue_site'].mean()),
                        'selective_full_rescue_rate': float(subset['selective_full_rescue_site'].mean()),
                    }
                )
            else:
                summary_rows.append(
                    {
                        'analysis_unit': analysis_unit,
                        'subset_label': subset_label,
                        'n': int(len(subset)),
                        'mean_always_tl_delta': float(subset['delta_abs_error_always_tl_vs_no_tl'].mean()),
                        'mean_selective_delta_vs_no_tl': float(subset['delta_abs_error_selective_vs_no_tl'].mean()),
                        'mean_selective_delta_vs_always_tl': float(subset['delta_abs_error_selective_vs_always_tl'].mean()),
                        'always_tl_help_rate': float((subset['delta_abs_error_always_tl_vs_no_tl'] < 0).mean()),
                        'always_tl_harm_rate': float((subset['delta_abs_error_always_tl_vs_no_tl'] > 0).mean()),
                        'selective_partial_rescue_rate': float(subset['selective_partial_rescue'].mean()),
                        'selective_full_rescue_rate': float(subset['selective_full_rescue'].mean()),
                    }
                )

        binned = site_summary[['site_id', 'cosine_similarity', 'delta_rmse_always_tl_vs_no_tl', 'delta_rmse_selective_vs_no_tl', 'selective_partial_rescue_site', 'selective_full_rescue_site']].dropna().copy()
        if not binned.empty:
            n_bins = int(min(4, max(1, binned['cosine_similarity'].nunique())))
            binned['similarity_bin'] = pd.qcut(binned['cosine_similarity'], q=n_bins, duplicates='drop')
            binned['similarity_bin'] = binned['similarity_bin'].astype(str)
            bin_summary = (
                binned.groupby('similarity_bin', as_index=False)
                .agg(
                    n_sites=('site_id', 'nunique'),
                    mean_delta_rmse_always_tl_vs_no_tl=('delta_rmse_always_tl_vs_no_tl', 'mean'),
                    mean_delta_rmse_selective_vs_no_tl=('delta_rmse_selective_vs_no_tl', 'mean'),
                    always_tl_harmed_site_rate=('delta_rmse_always_tl_vs_no_tl', lambda s: float((np.asarray(s) > 0).mean())),
                    selective_partial_rescue_rate=('selective_partial_rescue_site', 'mean'),
                    selective_full_rescue_rate=('selective_full_rescue_site', 'mean'),
                )
            )
        else:
            bin_summary = pd.DataFrame()

        return site_summary, pd.DataFrame(summary_rows), bin_summary

    def run_label_budget_evaluation(
        self,
        budgets: list[int] | None = None,
        run_rf: bool = True,
        run_selective_tl: bool = True,
        run_baseline: bool = True,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        budgets = [12, 24, 48, 72] if budgets is None else budgets
        budget_values = sorted({int(budget) for budget in budgets if int(budget) > 0})

        detail_frames = []
        full_detail = self.results['summary_long'].copy()
        full_detail['budget_label'] = 'full'
        full_detail['budget_windows_per_site'] = np.nan
        full_detail['effective_train_windows_total'] = int(self.results.get('effective_train_windows_total', 0))
        full_detail['mean_train_windows_per_site'] = float(self.results['effective_train_window_summary']['train_windows_used'].mean())
        full_detail['median_train_windows_per_site'] = float(self.results['effective_train_window_summary']['train_windows_used'].median())
        detail_frames.append(full_detail)

        for budget in budget_values:
            trial = self._inherit_trial_configuration(LSTMTransferLearningImplementation(
                root=self.root,
                sequence_length=self.sequence_length,
                seed=self.seed,
            ))
            trial.train_window_budget = int(budget)
            trial_results = trial.run(
                run_rf=run_rf,
                run_selective_tl=run_selective_tl,
                run_baseline=run_baseline,
                run_significance=False,
                run_seed_stability=False,
                run_rolling_temporal=False,
                run_selector_ablation=False,
                run_label_budget=False,
                persist_artifacts=False,
                display_outputs=False,
            )
            summary = trial_results['summary_long'].copy()
            summary['budget_label'] = f'{budget}'
            summary['budget_windows_per_site'] = int(budget)
            summary['effective_train_windows_total'] = int(trial_results.get('effective_train_windows_total', 0))
            summary['mean_train_windows_per_site'] = float(trial_results['effective_train_window_summary']['train_windows_used'].mean())
            summary['median_train_windows_per_site'] = float(trial_results['effective_train_window_summary']['train_windows_used'].median())
            detail_frames.append(summary)

        detail_df = pd.concat(detail_frames, ignore_index=True)
        summary_df = detail_df[detail_df['split'] == 'test'].copy()
        summary_df['budget_sort'] = summary_df['budget_windows_per_site'].fillna(summary_df['mean_train_windows_per_site']).astype(float)
        full_lookup = (
            summary_df[summary_df['budget_label'] == 'full'][['model_family', 'rmse']]
            .rename(columns={'rmse': 'rmse_full_budget'})
        )
        summary_df = summary_df.merge(full_lookup, on='model_family', how='left')
        no_tl_lookup = (
            summary_df[summary_df['model_family'] == self.NO_TL_NAME][['budget_label', 'rmse']]
            .rename(columns={'rmse': 'rmse_no_tl_same_budget'})
        )
        always_tl_lookup = (
            summary_df[summary_df['model_family'] == self.ALWAYS_TL_NAME][['budget_label', 'rmse']]
            .rename(columns={'rmse': 'rmse_always_tl_same_budget'})
        )
        summary_df = summary_df.merge(no_tl_lookup, on='budget_label', how='left')
        summary_df = summary_df.merge(always_tl_lookup, on='budget_label', how='left')
        summary_df['delta_rmse_vs_full_budget'] = summary_df['rmse'] - summary_df['rmse_full_budget']
        summary_df['delta_rmse_vs_no_tl_same_budget'] = summary_df['rmse'] - summary_df['rmse_no_tl_same_budget']
        summary_df['delta_rmse_vs_always_tl_same_budget'] = summary_df['rmse'] - summary_df['rmse_always_tl_same_budget']

        key_models = [
            self.SITE_HARD_TL_NAME,
            self.SELECTIVE_TL_NAME,
            self.ALWAYS_TL_NAME,
            self.NO_TL_NAME,
            'Random Forest',
            self.BASELINE_NO_TL_NAME,
            self.BASELINE_TL_NAME,
        ]
        summary_df = summary_df[summary_df['model_family'].isin(key_models)].copy()
        return detail_df, summary_df.sort_values(['budget_sort', 'rmse', 'model_family']).reset_index(drop=True)

    def build_publication_core_comparison(self, summary_wide: pd.DataFrame) -> pd.DataFrame:
        core_models = [
            self.SITE_HARD_TL_NAME,
            self.ALWAYS_TL_NAME,
            self.SELECTIVE_TL_NAME,
            self.NO_TL_NAME,
            'Random Forest',
            self.BASELINE_NO_TL_NAME,
            self.BASELINE_TL_NAME,
        ]
        out = summary_wide[summary_wide['model_family'].isin(core_models)].copy()
        if out.empty:
            return out

        out['test_rank_rmse'] = out['rmse_test'].rank(method='dense')
        selective_reference = self.SITE_HARD_TL_NAME if self.SITE_HARD_TL_NAME in set(out['model_family']) else self.SELECTIVE_TL_NAME
        out['delta_rmse_vs_selective'] = out['rmse_test'] - float(
            out.loc[out['model_family'] == selective_reference, 'rmse_test'].iloc[0]
        )
        out['delta_mae_vs_selective'] = out['mae_test'] - float(
            out.loc[out['model_family'] == selective_reference, 'mae_test'].iloc[0]
        )
        out['selective_reference_model'] = selective_reference
        ordered_cols = [
            'model_family',
            'test_rank_rmse',
            'rmse_test',
            'mae_test',
            'r2_test',
            'delta_rmse_vs_selective',
            'delta_mae_vs_selective',
        ]
        if 'n_test' in out.columns:
            ordered_cols.append('n_test')
        return out[ordered_cols].sort_values(['test_rank_rmse', 'rmse_test', 'model_family']).reset_index(drop=True)

    def build_publication_robustness_comparison(self) -> pd.DataFrame:
        core = self.results.get('publication_core_comparison')
        if core is None or core.empty:
            return pd.DataFrame()

        out = core[
            [
                'model_family',
                'test_rank_rmse',
                'rmse_test',
                'mae_test',
                'r2_test',
            ]
        ].copy()

        seed_summary = self.results.get('seed_stability_summary')
        if seed_summary is not None and not seed_summary.empty:
            seed_test = (
                seed_summary[seed_summary['split'] == 'test'][
                    ['model_family', 'n_seeds', 'mean_rmse', 'std_rmse', 'mean_mae', 'std_mae']
                ]
                .rename(
                    columns={
                        'n_seeds': 'seed_n',
                        'mean_rmse': 'seed_mean_rmse_test',
                        'std_rmse': 'seed_std_rmse_test',
                        'mean_mae': 'seed_mean_mae_test',
                        'std_mae': 'seed_std_mae_test',
                    }
                )
            )
            out = out.merge(seed_test, on='model_family', how='left')

        rolling_summary = self.results.get('rolling_temporal_summary')
        if rolling_summary is not None and not rolling_summary.empty:
            rolling_test = (
                rolling_summary[rolling_summary['split'] == 'test'][
                    ['model_family', 'n_folds', 'mean_rmse', 'std_rmse', 'mean_mae', 'std_mae']
                ]
                .rename(
                    columns={
                        'n_folds': 'rolling_n_folds',
                        'mean_rmse': 'rolling_mean_rmse_test',
                        'std_rmse': 'rolling_std_rmse_test',
                        'mean_mae': 'rolling_mean_mae_test',
                        'std_mae': 'rolling_std_mae_test',
                    }
                )
            )
            out = out.merge(rolling_test, on='model_family', how='left')

        return out.sort_values(['test_rank_rmse', 'rmse_test', 'model_family']).reset_index(drop=True)

    @staticmethod
    def _apply_split_months(df: pd.DataFrame, split_months: dict[str, pd.Index], default_label: str = 'unused') -> pd.DataFrame:
        out = df.copy()
        out['target_month'] = pd.to_datetime(out['target_month'])
        out['split'] = default_label
        for split_name, months in split_months.items():
            out.loc[out['target_month'].isin(pd.to_datetime(months)), 'split'] = split_name
        return out

    def build_rolling_temporal_folds(
        self,
        fold_count: int = 3,
        train_months: int = 120,
        val_months: int = 30,
        cal_months: int = 18,
        test_months: int = 30,
    ) -> tuple[list[dict], pd.DataFrame]:
        if self.tables is None:
            self.load_data()

        months = pd.Index(sorted(pd.to_datetime(self.tables['finetune_supervised']['target_month']).unique()))
        total_window = int(train_months + val_months + cal_months + test_months)
        if total_window > len(months):
            raise ValueError(
                f'Rolling temporal window ({total_window} months) exceeds available target history ({len(months)} months).'
            )

        max_start = len(months) - total_window
        start_positions = np.linspace(0, max_start, num=int(fold_count)).round().astype(int).tolist()
        start_positions = list(dict.fromkeys(int(pos) for pos in start_positions))

        fold_specs = []
        rows = []
        for fold_id, start in enumerate(start_positions, start=1):
            train_slice = months[start : start + train_months]
            val_slice = months[start + train_months : start + train_months + val_months]
            cal_slice = months[start + train_months + val_months : start + train_months + val_months + cal_months]
            test_slice = months[
                start + train_months + val_months + cal_months : start + total_window
            ]
            split_months = {
                'train': train_slice,
                'val': val_slice,
                'cal': cal_slice,
                'test': test_slice,
            }
            fold_specs.append({'fold_id': int(fold_id), 'split_months': split_months})
            rows.append(
                {
                    'fold_id': int(fold_id),
                    'start_index': int(start),
                    'train_months': int(len(train_slice)),
                    'val_months': int(len(val_slice)),
                    'cal_months': int(len(cal_slice)),
                    'test_months': int(len(test_slice)),
                    'train_start': train_slice.min(),
                    'train_end': train_slice.max(),
                    'val_start': val_slice.min(),
                    'val_end': val_slice.max(),
                    'cal_start': cal_slice.min(),
                    'cal_end': cal_slice.max(),
                    'test_start': test_slice.min(),
                    'test_end': test_slice.max(),
                }
            )

        return fold_specs, pd.DataFrame(rows)

    def _build_temporal_fold_tables(self, split_months: dict[str, pd.Index]) -> dict[str, object]:
        base_tables = self.tables if self.tables is not None else self.load_data()
        pretrain_supervised = self._apply_split_months(base_tables['pretrain_supervised'], split_months)
        finetune_supervised = self._apply_split_months(base_tables['finetune_supervised'], split_months)
        best_matches, source_target_similarity = DataLoader.build_similarity_artifacts(
            pretrain_supervised=pretrain_supervised,
            finetune_supervised=finetune_supervised,
            target_splits=('train',),
            persist=False,
        )

        tables = dict(base_tables)
        tables['pretrain_supervised'] = pretrain_supervised
        tables['finetune_supervised'] = finetune_supervised
        tables['best_matches'] = best_matches
        tables['source_target_similarity'] = source_target_similarity
        tables['similarity_target_splits'] = ['train']
        return tables

    def build_selector_features(self) -> pd.DataFrame:
        similarity = self.tables['source_target_similarity'][['site_id', 'cosine_similarity']].copy()

        similarity_summary = similarity.groupby('site_id').agg(
            sim_mean=('cosine_similarity', 'mean'),
            sim_std=('cosine_similarity', 'std'),
            sim_max=('cosine_similarity', 'max'),
            sim_min=('cosine_similarity', 'min'),
            sim_q90=('cosine_similarity', lambda s: float(s.quantile(0.90))),
            sim_q95=('cosine_similarity', lambda s: float(s.quantile(0.95))),
        ).reset_index()

        ranked = similarity.sort_values(['site_id', 'cosine_similarity'], ascending=[True, False]).copy()
        top_frames = []
        for k in [3, 5, 10]:
            topk = ranked.groupby('site_id').head(k)
            topk_summary = topk.groupby('site_id').agg(
                **{
                    f'sim_top{k}_mean': ('cosine_similarity', 'mean'),
                    f'sim_top{k}_std': ('cosine_similarity', 'std'),
                }
            ).reset_index()
            top_frames.append(topk_summary)

        train_df = self.tables['finetune_supervised'].copy()
        train_df = train_df[train_df['split'] == 'train'].copy()
        train_summary = train_df.groupby('site_id').agg(
            train_rows=('site_id', 'size'),
            wtda_mean=('wtda', 'mean'),
            wtda_std=('wtda', 'std'),
            pr_a_mean=('pr_a', 'mean'),
            pr_a_std=('pr_a', 'std'),
            sm_a_mean=('sm_a', 'mean'),
            sm_a_std=('sm_a', 'std'),
            tsmp_wtda_mean=('tsmp_wtda', 'mean'),
            tsmp_wtda_std=('tsmp_wtda', 'std'),
            pumping_log_mean=('pumping_m3_month_log1p', 'mean'),
            pumping_log_std=('pumping_m3_month_log1p', 'std'),
            lon_norm=('lon_norm', 'mean'),
            lat_norm=('lat_norm', 'mean'),
            max_observations=('max_observations', 'max'),
        ).reset_index()

        out = similarity_summary.merge(train_summary, on='site_id', how='left')
        for frame in top_frames:
            out = out.merge(frame, on='site_id', how='left')

        effective_train_window_summary = self.results.get('effective_train_window_summary')
        if effective_train_window_summary is not None and not effective_train_window_summary.empty:
            out = out.merge(
                effective_train_window_summary.rename(columns={'train_windows_used': 'train_rows_budgeted'}),
                on='site_id',
                how='left',
            )
            out['train_rows'] = out['train_rows_budgeted'].fillna(out['train_rows']).fillna(0.0)
            out = out.drop(columns='train_rows_budgeted')

        numeric_cols = [col for col in out.columns if col != 'site_id']
        out[numeric_cols] = out[numeric_cols].fillna(out[numeric_cols].median())
        return out

    def build_analysis_cohorts(self) -> tuple[pd.DataFrame, pd.DataFrame]:
        selector_meta = (
            self.results['selective_tl']['selector_by_site'][['site_id', 'cosine_similarity', 'months_used', 'train_rows']]
            .drop_duplicates('site_id')
            .copy()
        )

        sim_tau = float(selector_meta['cosine_similarity'].quantile(0.75))
        overlap_tau = float(selector_meta['months_used'].quantile(0.75))
        history_tau = float(selector_meta['train_rows'].quantile(0.75))

        selector_meta['all_wells'] = True
        selector_meta['high_similarity_q75'] = selector_meta['cosine_similarity'] >= sim_tau
        selector_meta['high_overlap_q75'] = selector_meta['months_used'] >= overlap_tau
        selector_meta['history_rich_q75'] = selector_meta['train_rows'] >= history_tau
        selector_meta['transfer_ready_q75'] = selector_meta['high_similarity_q75'] & selector_meta['high_overlap_q75']

        cohort_specs = [
            ('all_wells', 'All wells', 'Full Amsterdam test cohort', np.nan),
            ('high_similarity_q75', 'High similarity', 'Best-match cosine similarity >= 75th percentile', sim_tau),
            ('high_overlap_q75', 'High overlap', 'Best-match overlap months >= 75th percentile', overlap_tau),
            ('history_rich_q75', 'History-rich', 'Training rows >= 75th percentile', history_tau),
            ('transfer_ready_q75', 'Transfer-ready', 'High similarity and high overlap', np.nan),
        ]

        definition_rows = []
        for cohort_key, cohort_label, description, threshold in cohort_specs:
            mask = selector_meta[cohort_key].astype(bool)
            definition_rows.append(
                {
                    'cohort_key': cohort_key,
                    'cohort_label': cohort_label,
                    'description': description,
                    'threshold_value': threshold,
                    'n_wells': int(mask.sum()),
                }
            )

        assignment_rows = []
        for cohort_key, cohort_label, _, _ in cohort_specs:
            mask = selector_meta[cohort_key].astype(bool)
            cohort_sites = selector_meta.loc[mask, ['site_id', 'cosine_similarity', 'months_used', 'train_rows']].copy()
            cohort_sites['cohort_key'] = cohort_key
            cohort_sites['cohort_label'] = cohort_label
            assignment_rows.append(cohort_sites)

        return pd.DataFrame(definition_rows), pd.concat(assignment_rows, ignore_index=True)

    def build_cohort_site_summary(
        self,
        site_metric_wide: pd.DataFrame,
        cohort_assignments: pd.DataFrame,
    ) -> pd.DataFrame:
        cohort_map = cohort_assignments[['cohort_key', 'cohort_label', 'site_id']].drop_duplicates()
        model_cols = [col for col in site_metric_wide.columns if col != 'site_id']

        rows = []
        for cohort_key, cohort_frame in cohort_map.groupby('cohort_key'):
            cohort_label = cohort_frame['cohort_label'].iloc[0]
            site_ids = cohort_frame['site_id'].unique()
            subset = site_metric_wide[site_metric_wide['site_id'].isin(site_ids)].copy()
            if subset.empty:
                continue

            for model_name in model_cols:
                rows.append(
                    {
                        'cohort_key': cohort_key,
                        'cohort_label': cohort_label,
                        'model_family': model_name,
                        'n_wells': int(len(subset)),
                        'mean_site_rmse': float(subset[model_name].mean()),
                        'median_site_rmse': float(subset[model_name].median()),
                    }
                )

        out = pd.DataFrame(rows)
        if out.empty:
            return out

        out['rank_mean_site_rmse'] = out.groupby('cohort_key')['mean_site_rmse'].rank(method='dense')
        return out.sort_values(['cohort_key', 'rank_mean_site_rmse', 'mean_site_rmse']).reset_index(drop=True)

    def build_cohort_pooled_summary(
        self,
        model_payloads: dict[str, dict],
        cohort_assignments: pd.DataFrame,
        split: str = 'test',
    ) -> pd.DataFrame:
        cohort_map = cohort_assignments[['cohort_key', 'cohort_label', 'site_id']].drop_duplicates()
        rows = []

        for cohort_key, cohort_frame in cohort_map.groupby('cohort_key'):
            cohort_label = cohort_frame['cohort_label'].iloc[0]
            site_ids = set(cohort_frame['site_id'].tolist())

            for model_name, payload in model_payloads.items():
                frame = self._error_frame(payload, split)
                subset = frame[frame['site_id'].isin(site_ids)].copy()
                if subset.empty:
                    continue

                y_true = subset['y_true'].to_numpy(dtype=np.float32)
                y_pred = subset['y_pred'].to_numpy(dtype=np.float32)
                rows.append(
                    {
                        'cohort_key': cohort_key,
                        'cohort_label': cohort_label,
                        'split': split,
                        'model_family': model_name,
                        'n_wells': int(subset['site_id'].nunique()),
                        'n_observations': int(len(subset)),
                        'rmse': float(np.sqrt(np.mean((y_true - y_pred) ** 2))),
                        'mae': float(np.mean(np.abs(y_true - y_pred))),
                        'r2': float(r2_score(y_true, y_pred)),
                    }
                )

        out = pd.DataFrame(rows)
        if out.empty:
            return out

        out['rank_rmse'] = out.groupby('cohort_key')['rmse'].rank(method='dense')
        return out.sort_values(['cohort_key', 'rank_rmse', 'rmse']).reset_index(drop=True)

    def plot_cohort_site_summary(self, cohort_site_summary: pd.DataFrame) -> Path | None:
        if cohort_site_summary.empty:
            return None

        key_models = [
            self.SITE_HARD_TL_NAME,
            self.SELECTIVE_TL_NAME,
            self.ALWAYS_TL_NAME,
            self.NO_TL_NAME,
            'Random Forest',
            self.BASELINE_NO_TL_NAME,
            self.BASELINE_TL_NAME,
        ]
        plot_df = cohort_site_summary[cohort_site_summary['model_family'].isin(key_models)].copy()
        if plot_df.empty:
            return None

        cohort_order = ['All wells', 'High similarity', 'High overlap', 'History-rich', 'Transfer-ready']
        model_order = [model for model in key_models if model in plot_df['model_family'].unique()]

        fig, ax = plt.subplots(figsize=(11.5, 5.2))
        sns.barplot(
            data=plot_df,
            x='cohort_label',
            y='mean_site_rmse',
            hue='model_family',
            order=cohort_order,
            hue_order=model_order,
            ax=ax,
        )
        ax.set_title('Per-Well Test RMSE Across Pre-Specified Cohorts')
        ax.set_xlabel('')
        ax.set_ylabel('Mean site RMSE')
        ax.tick_params(axis='x', rotation=12)
        ax.legend(title='', fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1.02))
        fig.tight_layout()
        out_path = self.figure_dir / 'cohort_mean_site_rmse.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    def plot_no_tl_vs_selective(self, site_metric_wide: pd.DataFrame) -> Path | None:
        if self.NO_TL_NAME not in site_metric_wide.columns or self.SELECTIVE_TL_NAME not in site_metric_wide.columns:
            return None

        plot_df = site_metric_wide[['site_id', self.NO_TL_NAME, self.SELECTIVE_TL_NAME]].dropna().copy()
        fig, ax = plt.subplots(figsize=(6.5, 6))
        sns.scatterplot(data=plot_df, x=self.NO_TL_NAME, y=self.SELECTIVE_TL_NAME, color='#2a9d8f', ax=ax)
        min_axis = float(plot_df[[self.NO_TL_NAME, self.SELECTIVE_TL_NAME]].min().min())
        max_axis = float(plot_df[[self.NO_TL_NAME, self.SELECTIVE_TL_NAME]].max().max())
        ax.plot([min_axis, max_axis], [min_axis, max_axis], linestyle='--', color='gray', linewidth=1)
        ax.set_title('Site-Level RMSE: No TL vs Selective Learning')
        ax.set_xlabel(f'{self.NO_TL_NAME} RMSE')
        ax.set_ylabel(f'{self.SELECTIVE_TL_NAME} RMSE')
        fig.tight_layout()
        out_path = self.figure_dir / 'site_level_rmse_no_tl_vs_selective_tl.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    def plot_similarity_vs_gain(self, site_effects: pd.DataFrame) -> Path:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)

        always_rho = site_effects['cosine_similarity'].corr(site_effects['delta_rmse_always_tl_vs_no_tl'], method='spearman')
        selective_rho = site_effects['cosine_similarity'].corr(site_effects['delta_rmse_selective_tl_vs_no_tl'], method='spearman')

        sns.scatterplot(
            data=site_effects,
            x='cosine_similarity',
            y='delta_rmse_always_tl_vs_no_tl',
            color='#e76f51',
            ax=axes[0],
        )
        axes[0].axhline(0.0, color='gray', ls='--', lw=1)
        axes[0].set_title(f'Always TL Gain vs Similarity (rho={always_rho:.2f})')
        axes[0].set_xlabel('Best-match cosine similarity')
        axes[0].set_ylabel('Delta RMSE vs No TL')

        sns.scatterplot(
            data=site_effects,
            x='cosine_similarity',
            y='delta_rmse_selective_tl_vs_no_tl',
            color='#2a9d8f',
            ax=axes[1],
        )
        axes[1].axhline(0.0, color='gray', ls='--', lw=1)
        axes[1].set_title(f'Selective Learning Gain vs Similarity (rho={selective_rho:.2f})')
        axes[1].set_xlabel('Best-match cosine similarity')
        axes[1].set_ylabel('')

        fig.tight_layout()
        out_path = self.figure_dir / 'similarity_vs_transfer_gain.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    def plot_negative_transfer_bins(self, bin_summary: pd.DataFrame) -> Path | None:
        if bin_summary.empty:
            return None

        plot_df = bin_summary.copy()
        fig, ax1 = plt.subplots(figsize=(10.5, 4.8))
        plot_df = plot_df.melt(
            id_vars=['similarity_bin', 'n_sites', 'always_tl_harmed_site_rate', 'selective_partial_rescue_rate', 'selective_full_rescue_rate'],
            value_vars=['mean_delta_rmse_always_tl_vs_no_tl', 'mean_delta_rmse_selective_vs_no_tl'],
            var_name='strategy',
            value_name='mean_delta_rmse',
        )
        plot_df['strategy'] = plot_df['strategy'].map(
            {
                'mean_delta_rmse_always_tl_vs_no_tl': 'Always TL',
                'mean_delta_rmse_selective_vs_no_tl': 'Selective Learning',
            }
        )
        sns.barplot(
            data=plot_df,
            x='similarity_bin',
            y='mean_delta_rmse',
            hue='strategy',
            ax=ax1,
        )
        ax1.axhline(0.0, color='gray', ls='--', lw=1)
        ax1.set_xlabel('Similarity quartile')
        ax1.set_ylabel('Mean delta RMSE vs No TL')
        ax1.set_title('Negative Transfer by Source-Target Similarity')
        ax1.tick_params(axis='x', rotation=10)
        ax1.legend(title='', loc='upper left')

        ax2 = ax1.twinx()
        rescue_df = bin_summary.copy()
        ax2.plot(
            np.arange(len(rescue_df)),
            rescue_df['selective_full_rescue_rate'],
            color='#1d3557',
            marker='o',
            linewidth=1.8,
        )
        ax2.set_ylabel('Selective full-rescue rate')
        ax2.set_ylim(0.0, 1.0)

        out_path = self.figure_dir / 'negative_transfer_similarity_bins.png'
        fig.tight_layout()
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    def plot_selector_ablation(self, ablation_summary: pd.DataFrame) -> Path | None:
        if ablation_summary.empty:
            return None

        plot_df = ablation_summary.copy().sort_values('rmse_test')
        fig, ax = plt.subplots(figsize=(8.8, 4.8))
        sns.barplot(data=plot_df, x='rmse_test', y='ablation_label', color='#457b9d', ax=ax)
        ax.set_title('Selective-Learning Ablation on Test RMSE')
        ax.set_xlabel('Test RMSE')
        ax.set_ylabel('')
        fig.tight_layout()
        out_path = self.figure_dir / 'selector_ablation_test_rmse.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    def plot_label_budget_curve(self, label_budget_summary: pd.DataFrame) -> Path | None:
        if label_budget_summary.empty:
            return None

        plot_df = label_budget_summary.copy()
        model_order = [
            self.SITE_HARD_TL_NAME,
            self.SELECTIVE_TL_NAME,
            self.ALWAYS_TL_NAME,
            self.NO_TL_NAME,
            'Random Forest',
            self.BASELINE_NO_TL_NAME,
            self.BASELINE_TL_NAME,
        ]
        plot_df = plot_df[plot_df['model_family'].isin(model_order)].copy()
        if plot_df.empty:
            return None

        fig, ax = plt.subplots(figsize=(10.8, 5.2))
        sns.lineplot(
            data=plot_df,
            x='mean_train_windows_per_site',
            y='rmse',
            hue='model_family',
            style='model_family',
            markers=True,
            dashes=False,
            hue_order=[model for model in model_order if model in plot_df['model_family'].unique()],
            ax=ax,
        )
        ax.set_title('Label-Budget Stress Test')
        ax.set_xlabel('Mean target train windows per site')
        ax.set_ylabel('Test RMSE')
        ax.legend(title='', fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1.02))
        fig.tight_layout()
        out_path = self.figure_dir / 'label_budget_curve.png'
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    def plot_selector_candidates(self, selector_search: pd.DataFrame) -> Path | None:
        if selector_search.empty:
            return None

        plot_df = selector_search.copy()
        fig, ax1 = plt.subplots(figsize=(8.4, 4.8))
        sns.barplot(data=plot_df, x='candidate_model', y='mean_site_rmse_oof', color='#2a9d8f', ax=ax1)
        ax1.set_xlabel('')
        ax1.set_ylabel('OOF mean site RMSE', color='#2a9d8f')
        ax1.tick_params(axis='y', labelcolor='#2a9d8f')
        ax1.tick_params(axis='x', rotation=10)
        ax1.set_title('Selective Learning Gate Search on Calibration OOF')

        ax2 = ax1.twinx()
        ax2.plot(
            np.arange(len(plot_df)),
            plot_df['share_transfer_oof'],
            color='#e76f51',
            marker='o',
            linewidth=1.8,
        )
        ax2.set_ylabel('OOF transfer share', color='#e76f51')
        ax2.tick_params(axis='y', labelcolor='#e76f51')

        out_path = self.figure_dir / 'selector_tradeoff_curve.png'
        fig.tight_layout()
        fig.savefig(out_path, dpi=180, bbox_inches='tight')
        plt.close(fig)
        return out_path

    @staticmethod
    def _error_frame(payload: dict, split: str) -> pd.DataFrame:
        meta = payload[f'{split}_meta'][['site_id', 'target_month']].copy().reset_index(drop=True)
        frame = meta.copy()
        frame['y_true'] = np.asarray(payload[f'{split}_true']).flatten()
        frame['y_pred'] = np.asarray(payload[f'{split}_pred']).flatten()
        frame['abs_error'] = np.abs(frame['y_true'] - frame['y_pred'])
        return frame.sort_values(['site_id', 'target_month']).reset_index(drop=True)

    @staticmethod
    def _holm_adjust(p_values: list[float]) -> list[float]:
        m = len(p_values)
        indexed = sorted(enumerate(p_values), key=lambda item: item[1])
        adjusted = [1.0] * m
        running_max = 0.0
        for rank, (idx, p_value) in enumerate(indexed, start=1):
            scaled = min(1.0, (m - rank + 1) * float(p_value))
            running_max = max(running_max, scaled)
            adjusted[idx] = running_max
        return adjusted

    @staticmethod
    def _rank_biserial(diff: np.ndarray) -> float:
        from scipy.stats import rankdata

        diff = np.asarray(diff, dtype=np.float64)
        diff = diff[diff != 0]
        if diff.size == 0:
            return 0.0

        ranks = rankdata(np.abs(diff))
        positive = float(ranks[diff > 0].sum())
        negative = float(ranks[diff < 0].sum())
        denom = positive + negative
        if denom == 0:
            return 0.0
        return (positive - negative) / denom

    @staticmethod
    def _sign_test_p_value(wins_model_a: int, wins_model_b: int) -> float:
        from scipy.stats import binomtest

        n_trials = int(wins_model_a) + int(wins_model_b)
        if n_trials == 0:
            return 1.0
        return float(binomtest(int(wins_model_a), n=n_trials, p=0.5, alternative='two-sided').pvalue)

    def _paired_test_row(
        self,
        split: str,
        analysis_unit: str,
        model_a: str,
        model_b: str,
        metric_a: np.ndarray,
        metric_b: np.ndarray,
    ) -> dict:
        from scipy.stats import wilcoxon

        metric_a = np.asarray(metric_a, dtype=np.float64)
        metric_b = np.asarray(metric_b, dtype=np.float64)
        diff = metric_a - metric_b
        non_zero = diff[np.abs(diff) > 1e-12]
        if non_zero.size == 0:
            statistic = 0.0
            p_value = 1.0
            rank_biserial = 0.0
        else:
            test = wilcoxon(non_zero, zero_method='wilcox', alternative='two-sided', method='auto')
            statistic = float(test.statistic)
            p_value = float(test.pvalue)
            rank_biserial = float(self._rank_biserial(non_zero))

        wins_model_a = int((diff < 0).sum())
        wins_model_b = int((diff > 0).sum())
        ties = int(np.isclose(diff, 0.0).sum())
        mean_a = float(metric_a.mean())
        mean_b = float(metric_b.mean())
        return {
            'split': split,
            'analysis_unit': analysis_unit,
            'model_a': model_a,
            'model_b': model_b,
            'n_pairs': int(diff.size),
            'metric_mean_a': mean_a,
            'metric_mean_b': mean_b,
            'mean_delta_metric_a_minus_b': float(diff.mean()),
            'median_delta_metric_a_minus_b': float(np.median(diff)),
            'wins_model_a': wins_model_a,
            'wins_model_b': wins_model_b,
            'ties': ties,
            'sign_test_p_value': self._sign_test_p_value(wins_model_a, wins_model_b),
            'wilcoxon_statistic': statistic,
            'p_value': p_value,
            'rank_biserial_a_minus_b': rank_biserial,
            'better_model': model_a if mean_a < mean_b else model_b,
        }

    def compute_significance_tests(
        self,
        model_payloads: dict[str, dict],
        split: str = 'test',
        comparisons: list[tuple[str, str]] | None = None,
    ) -> pd.DataFrame:
        if comparisons is None:
            main_selective = self.SITE_HARD_TL_NAME
            comparisons = [
                (main_selective, 'Random Forest'),
                (main_selective, self.SELECTIVE_TL_NAME),
                (main_selective, self.NO_TL_NAME),
                (main_selective, self.ALWAYS_TL_NAME),
                (main_selective, self.BASELINE_NO_TL_NAME),
                (main_selective, self.BASELINE_TL_NAME),
            ]

        rows = []
        for model_a, model_b in comparisons:
            if model_a not in model_payloads or model_b not in model_payloads:
                continue

            a_frame = self._error_frame(model_payloads[model_a], split).rename(columns={'abs_error': 'abs_error_a'})
            b_frame = self._error_frame(model_payloads[model_b], split).rename(columns={'abs_error': 'abs_error_b'})
            merged = a_frame[['site_id', 'target_month', 'abs_error_a']].merge(
                b_frame[['site_id', 'target_month', 'abs_error_b']],
                on=['site_id', 'target_month'],
                how='inner',
            )
            if merged.empty:
                continue

            rows.append(
                self._paired_test_row(
                    split=split,
                    analysis_unit='observation_abs_error',
                    model_a=model_a,
                    model_b=model_b,
                    metric_a=merged['abs_error_a'].to_numpy(dtype=np.float64),
                    metric_b=merged['abs_error_b'].to_numpy(dtype=np.float64),
                )
            )

            site_a = self._site_level_rmse(
                model_payloads[model_a][f'{split}_true'],
                model_payloads[model_a][f'{split}_pred'],
                model_payloads[model_a][f'{split}_meta'],
                'site_metric_a',
            )
            site_b = self._site_level_rmse(
                model_payloads[model_b][f'{split}_true'],
                model_payloads[model_b][f'{split}_pred'],
                model_payloads[model_b][f'{split}_meta'],
                'site_metric_b',
            )
            site_merged = site_a.merge(site_b, on='site_id', how='inner')
            if not site_merged.empty:
                rows.append(
                    self._paired_test_row(
                        split=split,
                        analysis_unit='site_rmse',
                        model_a=model_a,
                        model_b=model_b,
                        metric_a=site_merged['site_metric_a'].to_numpy(dtype=np.float64),
                        metric_b=site_merged['site_metric_b'].to_numpy(dtype=np.float64),
                    )
                )

        out = pd.DataFrame(rows)
        if out.empty:
            return out

        out['p_value_holm'] = 1.0
        out['sign_test_p_value_holm'] = 1.0
        for analysis_unit, idx in out.groupby('analysis_unit').groups.items():
            idx = list(idx)
            out.loc[idx, 'p_value_holm'] = self._holm_adjust(out.loc[idx, 'p_value'].tolist())
            out.loc[idx, 'sign_test_p_value_holm'] = self._holm_adjust(out.loc[idx, 'sign_test_p_value'].tolist())
        return out.sort_values(['analysis_unit', 'p_value', 'mean_delta_metric_a_minus_b']).reset_index(drop=True)

    def compute_cluster_bootstrap_tests(
        self,
        model_payloads: dict[str, dict],
        split: str = 'test',
        comparisons: list[tuple[str, str]] | None = None,
        n_bootstrap: int = 5000,
    ) -> pd.DataFrame:
        if comparisons is None:
            main_selective = self.SITE_HARD_TL_NAME
            comparisons = [
                (main_selective, 'Random Forest'),
                (main_selective, self.SELECTIVE_TL_NAME),
                (main_selective, self.NO_TL_NAME),
                (main_selective, self.ALWAYS_TL_NAME),
                (main_selective, self.BASELINE_NO_TL_NAME),
                (main_selective, self.BASELINE_TL_NAME),
            ]

        rng = np.random.default_rng(self.seed)
        rows = []
        for model_a, model_b in comparisons:
            if model_a not in model_payloads or model_b not in model_payloads:
                continue

            site_a = self._site_level_rmse(
                model_payloads[model_a][f'{split}_true'],
                model_payloads[model_a][f'{split}_pred'],
                model_payloads[model_a][f'{split}_meta'],
                'site_metric_a',
            )
            site_b = self._site_level_rmse(
                model_payloads[model_b][f'{split}_true'],
                model_payloads[model_b][f'{split}_pred'],
                model_payloads[model_b][f'{split}_meta'],
                'site_metric_b',
            )
            merged = site_a.merge(site_b, on='site_id', how='inner')
            if merged.empty:
                continue

            site_diff = (
                merged['site_metric_a'].to_numpy(dtype=np.float64)
                - merged['site_metric_b'].to_numpy(dtype=np.float64)
            )
            n_sites = int(site_diff.size)
            bootstrap_mean_diff = np.empty(n_bootstrap, dtype=np.float64)
            for i in range(n_bootstrap):
                sample_idx = rng.integers(0, n_sites, size=n_sites)
                bootstrap_mean_diff[i] = float(site_diff[sample_idx].mean())

            wins_model_a = int((site_diff < 0).sum())
            wins_model_b = int((site_diff > 0).sum())
            rows.append(
                {
                    'split': split,
                    'analysis_unit': 'site_rmse_cluster_bootstrap',
                    'model_a': model_a,
                    'model_b': model_b,
                    'n_sites': n_sites,
                    'n_bootstrap': int(n_bootstrap),
                    'site_metric_mean_a': float(merged['site_metric_a'].mean()),
                    'site_metric_mean_b': float(merged['site_metric_b'].mean()),
                    'mean_delta_site_rmse_a_minus_b': float(site_diff.mean()),
                    'median_delta_site_rmse_a_minus_b': float(np.median(site_diff)),
                    'wins_model_a': wins_model_a,
                    'wins_model_b': wins_model_b,
                    'ties': int(np.isclose(site_diff, 0.0).sum()),
                    'bootstrap_ci_lower': float(np.quantile(bootstrap_mean_diff, 0.025)),
                    'bootstrap_ci_upper': float(np.quantile(bootstrap_mean_diff, 0.975)),
                    'bootstrap_p_value': float(
                        2.0 * min((bootstrap_mean_diff <= 0.0).mean(), (bootstrap_mean_diff >= 0.0).mean())
                    ),
                    'pr_model_a_better': float((bootstrap_mean_diff < 0.0).mean()),
                    'better_model': model_a if merged['site_metric_a'].mean() < merged['site_metric_b'].mean() else model_b,
                }
            )

        out = pd.DataFrame(rows)
        if out.empty:
            return out
        out['bootstrap_p_value_holm'] = self._holm_adjust(out['bootstrap_p_value'].tolist())
        return out.sort_values(['bootstrap_p_value', 'mean_delta_site_rmse_a_minus_b']).reset_index(drop=True)

    def run_seed_stability(
        self,
        seeds: list[int],
        run_rf: bool = True,
        run_selective_tl: bool = True,
        run_baseline: bool = True,
    ) -> tuple[pd.DataFrame, pd.DataFrame]:
        detail_frames = []

        for seed in seeds:
            trial = self._inherit_trial_configuration(LSTMTransferLearningImplementation(
                root=self.root,
                sequence_length=self.sequence_length,
                seed=int(seed),
            ))
            trial_results = trial.run(
                run_rf=run_rf,
                run_selective_tl=run_selective_tl,
                run_baseline=run_baseline,
                run_significance=False,
                run_seed_stability=False,
                run_rolling_temporal=False,
                run_selector_ablation=False,
                run_label_budget=False,
                persist_artifacts=False,
                display_outputs=False,
            )
            summary = trial_results['summary_long'].copy()
            summary['seed'] = int(seed)
            detail_frames.append(summary)

        detail_df = pd.concat(detail_frames, ignore_index=True)
        summary_df = (
            detail_df.groupby(['model_family', 'split'], as_index=False)
            .agg(
                n_seeds=('seed', 'nunique'),
                mean_rmse=('rmse', 'mean'),
                std_rmse=('rmse', 'std'),
                mean_mae=('mae', 'mean'),
                std_mae=('mae', 'std'),
                mean_r2=('r2', 'mean'),
                std_r2=('r2', 'std'),
            )
        )
        std_cols = [col for col in summary_df.columns if col.startswith('std_')]
        summary_df[std_cols] = summary_df[std_cols].fillna(0.0)

        test_order = (
            summary_df[summary_df['split'] == 'test'][['model_family', 'mean_rmse']]
            .rename(columns={'mean_rmse': 'sort_rmse'})
        )
        summary_df = (
            summary_df.merge(test_order, on='model_family', how='left')
            .sort_values(['sort_rmse', 'split', 'model_family'])
            .drop(columns='sort_rmse')
            .reset_index(drop=True)
        )
        return detail_df, summary_df

    def run_rolling_temporal_evaluation(
        self,
        fold_count: int = 3,
        train_months: int = 120,
        val_months: int = 30,
        cal_months: int = 18,
        test_months: int = 30,
        run_rf: bool = True,
        run_selective_tl: bool = True,
        run_baseline: bool = True,
    ) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
        fold_specs, fold_definitions = self.build_rolling_temporal_folds(
            fold_count=fold_count,
            train_months=train_months,
            val_months=val_months,
            cal_months=cal_months,
            test_months=test_months,
        )

        detail_frames = []
        for spec in fold_specs:
            fold_tables = self._build_temporal_fold_tables(spec['split_months'])
            trial = self._inherit_trial_configuration(LSTMTransferLearningImplementation(
                root=self.root,
                sequence_length=self.sequence_length,
                seed=self.seed,
            ))
            trial_results = trial.run(
                run_rf=run_rf,
                run_selective_tl=run_selective_tl,
                run_baseline=run_baseline,
                run_significance=False,
                run_seed_stability=False,
                run_rolling_temporal=False,
                run_selector_ablation=False,
                run_label_budget=False,
                persist_artifacts=False,
                display_outputs=False,
                tables_override=fold_tables,
            )
            summary = trial_results['summary_long'].copy()
            fold_meta = fold_definitions[fold_definitions['fold_id'] == spec['fold_id']].iloc[0]
            summary['fold_id'] = int(spec['fold_id'])
            summary['train_start'] = fold_meta['train_start']
            summary['train_end'] = fold_meta['train_end']
            summary['test_start'] = fold_meta['test_start']
            summary['test_end'] = fold_meta['test_end']
            detail_frames.append(summary)

        detail_df = pd.concat(detail_frames, ignore_index=True)
        summary_df = (
            detail_df.groupby(['model_family', 'split'], as_index=False)
            .agg(
                n_folds=('fold_id', 'nunique'),
                mean_rmse=('rmse', 'mean'),
                std_rmse=('rmse', 'std'),
                min_rmse=('rmse', 'min'),
                max_rmse=('rmse', 'max'),
                mean_mae=('mae', 'mean'),
                std_mae=('mae', 'std'),
                mean_r2=('r2', 'mean'),
                std_r2=('r2', 'std'),
            )
        )
        std_cols = [col for col in summary_df.columns if col.startswith('std_')]
        summary_df[std_cols] = summary_df[std_cols].fillna(0.0)

        test_order = (
            summary_df[summary_df['split'] == 'test'][['model_family', 'mean_rmse']]
            .rename(columns={'mean_rmse': 'sort_rmse'})
        )
        summary_df = (
            summary_df.merge(test_order, on='model_family', how='left')
            .sort_values(['sort_rmse', 'split', 'model_family'])
            .drop(columns='sort_rmse')
            .reset_index(drop=True)
        )
        return fold_definitions, detail_df, summary_df


    @staticmethod
    def _error_summary(group: pd.DataFrame) -> pd.Series:
        y_true = group['y_true'].to_numpy(dtype=np.float64)
        y_pred = group['y_pred'].to_numpy(dtype=np.float64)
        residual = group['residual'].to_numpy(dtype=np.float64)
        abs_error = np.abs(residual)
        return pd.Series(
            {
                'n': int(len(group)),
                'rmse': float(np.sqrt(np.mean(np.square(residual)))) if len(group) else np.nan,
                'mae': float(np.mean(abs_error)) if len(group) else np.nan,
                'bias': float(np.mean(residual)) if len(group) else np.nan,
                'median_abs_error': float(np.median(abs_error)) if len(group) else np.nan,
                'p90_abs_error': float(np.quantile(abs_error, 0.90)) if len(group) else np.nan,
                'target_mean': float(np.mean(y_true)) if len(group) else np.nan,
                'target_std': float(np.std(y_true, ddof=0)) if len(group) else np.nan,
            }
        )

    def build_prediction_error_detail(self, model_payloads: dict[str, dict], split: str = 'test') -> pd.DataFrame:
        frames = []
        for model_name, payload in model_payloads.items():
            pred_key = f'{split}_pred'
            true_key = f'{split}_true'
            meta_key = f'{split}_meta'
            if pred_key not in payload or true_key not in payload or meta_key not in payload:
                continue

            frame = payload[meta_key].reset_index(drop=True).copy()
            frame['model_family'] = model_name
            frame['y_true'] = np.asarray(payload[true_key]).reshape(-1).astype(np.float32)
            frame['y_pred'] = np.asarray(payload[pred_key]).reshape(-1).astype(np.float32)
            frame['residual'] = frame['y_pred'] - frame['y_true']
            frame['abs_error'] = frame['residual'].abs()
            frame['sq_error'] = np.square(frame['residual'])
            frames.append(frame)

        if not frames:
            return pd.DataFrame()

        detail = pd.concat(frames, ignore_index=True)
        detail['target_month'] = pd.to_datetime(detail['target_month'])
        detail['target_year'] = detail['target_month'].dt.year
        detail['target_month_of_year'] = detail['target_month'].dt.month
        detail['target_abs'] = detail['y_true'].abs()
        detail['target_abs_regime'] = pd.cut(
            detail['target_abs'],
            bins=[-np.inf, 0.5, 1.5, np.inf],
            labels=['near-normal |y| <= 0.5', 'moderate 0.5 < |y| <= 1.5', 'extreme |y| > 1.5'],
            ordered=True,
        )
        detail['signed_anomaly_regime'] = pd.cut(
            detail['y_true'],
            bins=[-np.inf, -1.5, -0.5, 0.5, 1.5, np.inf],
            labels=['strong negative', 'moderate negative', 'near-normal', 'moderate positive', 'strong positive'],
            ordered=True,
        )
        detail['season'] = np.select(
            [
                detail['target_month_of_year'].isin([12, 1, 2]),
                detail['target_month_of_year'].isin([3, 4, 5]),
                detail['target_month_of_year'].isin([6, 7, 8]),
                detail['target_month_of_year'].isin([9, 10, 11]),
            ],
            ['winter', 'spring', 'summer', 'autumn'],
            default='unknown',
        )
        return detail.sort_values(['model_family', 'site_id', 'target_month']).reset_index(drop=True)

    def build_error_analysis_tables(self, error_detail: pd.DataFrame, site_metric_wide: pd.DataFrame) -> dict[str, pd.DataFrame]:
        if error_detail.empty:
            return {
                'error_by_anomaly_regime': pd.DataFrame(),
                'error_by_signed_anomaly_regime': pd.DataFrame(),
                'error_by_calendar_month': pd.DataFrame(),
                'site_error_diagnostics': pd.DataFrame(),
                'selector_failure_typology': pd.DataFrame(),
                'error_driver_correlations': pd.DataFrame(),
                'worst_site_error_profile': pd.DataFrame(),
            }

        error_by_anomaly = (
            error_detail.groupby(['model_family', 'target_abs_regime'], observed=True)
            .apply(self._error_summary)
            .reset_index()
            .sort_values(['target_abs_regime', 'rmse', 'model_family'])
        )
        error_by_signed = (
            error_detail.groupby(['model_family', 'signed_anomaly_regime'], observed=True)
            .apply(self._error_summary)
            .reset_index()
            .sort_values(['signed_anomaly_regime', 'rmse', 'model_family'])
        )
        error_by_month = (
            error_detail.groupby(['model_family', 'target_month_of_year'], observed=True)
            .apply(self._error_summary)
            .reset_index()
            .sort_values(['target_month_of_year', 'rmse', 'model_family'])
        )

        main_model = self.SITE_HARD_TL_NAME if self.SITE_HARD_TL_NAME in set(error_detail['model_family']) else self.SELECTIVE_TL_NAME
        main_error = error_detail[error_detail['model_family'] == main_model].copy()
        site_diag = (
            main_error.groupby('site_id', observed=True)
            .apply(self._error_summary)
            .reset_index()
            .rename(columns={
                'n': 'n_test',
                'rmse': 'site_hard_rmse',
                'mae': 'site_hard_mae',
                'bias': 'site_hard_bias',
                'median_abs_error': 'site_hard_median_abs_error',
                'p90_abs_error': 'site_hard_p90_abs_error',
                'target_std': 'test_target_std',
                'target_mean': 'test_target_mean',
            })
        )
        target_extra = (
            main_error.groupby('site_id', observed=True)
            .agg(
                mean_abs_target=('target_abs', 'mean'),
                max_abs_target=('target_abs', 'max'),
                extreme_share_abs_gt_1_5=('target_abs', lambda s: float((np.asarray(s) > 1.5).mean())),
                max_abs_error=('abs_error', 'max'),
            )
            .reset_index()
        )
        site_diag = site_diag.merge(target_extra, on='site_id', how='left')
        site_diag = site_diag.merge(site_metric_wide, on='site_id', how='left')

        route_cols = [col for col in ['site_id', 'selected_parent', 'use_transfer', 'cosine_similarity', 'months_used'] if col in main_error.columns]
        if len(route_cols) > 1:
            route_meta = main_error[route_cols].drop_duplicates('site_id')
            site_diag = site_diag.merge(route_meta, on='site_id', how='left')

        selector_meta = None
        site_hard_payload = self.results.get('site_hard_selective_tl', {})
        if isinstance(site_hard_payload, dict):
            selector_meta = site_hard_payload.get('selector_by_site')
        if isinstance(selector_meta, pd.DataFrame) and not selector_meta.empty:
            candidate_cols = ['site_id', 'selected_parent', 'use_transfer', 'cosine_similarity', 'months_used']
            keep = [col for col in candidate_cols if col in selector_meta.columns and (col == 'site_id' or col not in site_diag.columns)]
            if len(keep) > 1:
                site_diag = site_diag.merge(selector_meta[keep], on='site_id', how='left')

        soft_selector = self.results.get('selective_tl', {})
        soft_meta = soft_selector.get('selector_by_site') if isinstance(soft_selector, dict) else None
        if isinstance(soft_meta, pd.DataFrame) and not soft_meta.empty:
            keep = [
                col for col in [
                    'site_id', 'train_rows', 'max_observations', 'wtda_std', 'sim_top3_mean', 'selector_gamma_mean',
                    'selector_rmse_no_tl', 'selector_rmse_always_tl', 'selector_rmse_selective_learning_oof'
                ] if col in soft_meta.columns
            ]
            site_diag = site_diag.merge(soft_meta[keep], on='site_id', how='left')

        if self.NO_TL_NAME in site_diag.columns and self.ALWAYS_TL_NAME in site_diag.columns:
            no_rmse = site_diag[self.NO_TL_NAME]
            tl_rmse = site_diag[self.ALWAYS_TL_NAME]
            site_diag['test_best_parent'] = np.where(no_rmse <= tl_rmse, self.NO_TL_NAME, self.ALWAYS_TL_NAME)
            site_diag['test_best_parent_rmse'] = np.minimum(no_rmse, tl_rmse)
            site_diag['rmse_gap_vs_test_best_parent'] = site_diag['site_hard_rmse'] - site_diag['test_best_parent_rmse']
            site_diag['selection_matches_test_best_parent'] = site_diag['selected_parent'].eq(site_diag['test_best_parent'])
            site_diag['selector_failure_type'] = np.select(
                [
                    np.isclose(no_rmse, tl_rmse, atol=1e-6),
                    site_diag['selection_matches_test_best_parent'],
                ],
                ['parent_tie_on_test', 'selected_test_best_parent'],
                default='selected_test_worse_parent',
            )

        site_diag = site_diag.sort_values(['site_hard_rmse', 'site_id'], ascending=[False, True]).reset_index(drop=True)
        worst_sites = site_diag.head(12).copy()

        if 'selector_failure_type' in site_diag.columns:
            selector_failure = (
                site_diag.groupby(['selected_parent', 'test_best_parent', 'selector_failure_type'], dropna=False)
                .agg(
                    sites=('site_id', 'nunique'),
                    mean_site_hard_rmse=('site_hard_rmse', 'mean'),
                    mean_gap_vs_test_best_parent=('rmse_gap_vs_test_best_parent', 'mean'),
                    median_gap_vs_test_best_parent=('rmse_gap_vs_test_best_parent', 'median'),
                )
                .reset_index()
            )
            selector_failure['share_of_sites'] = selector_failure['sites'] / selector_failure['sites'].sum()
            selector_failure = selector_failure.sort_values(['selector_failure_type', 'sites'], ascending=[True, False])
        else:
            selector_failure = pd.DataFrame()

        driver_rows = []
        driver_cols = [
            'test_target_std',
            'mean_abs_target',
            'max_abs_target',
            'extreme_share_abs_gt_1_5',
            'train_rows',
            'max_observations',
            'cosine_similarity',
            'months_used',
            'wtda_std',
            'sim_top3_mean',
            'selector_gamma_mean',
        ]
        try:
            from scipy.stats import spearmanr
        except Exception:
            spearmanr = None
        for col in driver_cols:
            if col not in site_diag.columns:
                continue
            pair = site_diag[['site_hard_rmse', col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(pair) < 5 or pair[col].nunique() < 2:
                continue
            if spearmanr is not None:
                rho, p_value = spearmanr(pair[col].to_numpy(dtype=np.float64), pair['site_hard_rmse'].to_numpy(dtype=np.float64))
            else:
                rho = pair[col].corr(pair['site_hard_rmse'], method='spearman')
                p_value = np.nan
            driver_rows.append(
                {
                    'driver': col,
                    'n_sites': int(len(pair)),
                    'spearman_rho_with_site_hard_rmse': float(rho),
                    'p_value': float(p_value) if pd.notna(p_value) else np.nan,
                    'abs_spearman_rho': float(abs(rho)) if pd.notna(rho) else np.nan,
                }
            )
        driver_corr = pd.DataFrame(driver_rows).sort_values('abs_spearman_rho', ascending=False).reset_index(drop=True)

        return {
            'error_by_anomaly_regime': error_by_anomaly.reset_index(drop=True),
            'error_by_signed_anomaly_regime': error_by_signed.reset_index(drop=True),
            'error_by_calendar_month': error_by_month.reset_index(drop=True),
            'site_error_diagnostics': site_diag.reset_index(drop=True),
            'selector_failure_typology': selector_failure.reset_index(drop=True),
            'error_driver_correlations': driver_corr.reset_index(drop=True),
            'worst_site_error_profile': worst_sites.reset_index(drop=True),
        }

    def run(
        self,
        run_rf: bool = True,
        run_selective_tl: bool = True,
        run_baseline: bool = True,
        run_significance: bool = True,
        run_seed_stability: bool = True,
        run_rolling_temporal: bool = False,
        run_selector_ablation: bool = False,
        run_label_budget: bool = False,
        label_budget_windows: list[int] | None = None,
        stability_seeds: list[int] | None = None,
        rolling_fold_count: int = 3,
        rolling_train_months: int = 120,
        rolling_val_months: int = 30,
        rolling_cal_months: int = 18,
        rolling_test_months: int = 30,
        persist_artifacts: bool = True,
        display_outputs: bool = True,
        tables_override: dict[str, object] | None = None,
    ):
        if tables_override is not None:
            self.tables = tables_override
        else:
            self.load_data()
        self.prepare_datasets()

        protocol = self.summarize_protocol()
        if persist_artifacts:
            ModelEvaluator.save_table(protocol['protocol_summary'], result_table_path('protocol_summary.csv'))
            ModelEvaluator.save_table(protocol['feature_inventory'], result_table_path('feature_inventory.csv'))
            ModelEvaluator.save_table(protocol['similarity_summary'], result_table_path('similarity_summary.csv'))
            ModelEvaluator.save_table(protocol['transfer_configuration'], result_table_path('transfer_configuration.csv'))

        model_payloads = {}
        metrics = []

        if run_rf:
            rf_results = self.run_random_forest()
            self.results['random_forest'] = rf_results
            model_payloads['Random Forest'] = rf_results
            metrics.extend(rf_results['metrics'])

        if run_baseline:
            baseline_payloads = self.run_baseline_models()
            self.results['baseline_payloads'] = baseline_payloads
            for name, payload in baseline_payloads.items():
                model_payloads[name] = payload
                metrics.extend(payload['metrics'])

        if run_selective_tl:
            parent_payloads = self.run_parent_models()
            self.results['parent_payloads'] = parent_payloads
            selective_results = self.run_similarity_selector(compute_feature_importance=persist_artifacts)
            site_hard_results = self.run_site_hard_selector()

            self.results['selective_tl'] = selective_results
            self.results['site_hard_selective_tl'] = site_hard_results

            for name, payload in parent_payloads.items():
                model_payloads[name] = payload
                metrics.extend(payload['metrics'])

            model_payloads[self.SELECTIVE_TL_NAME] = selective_results
            model_payloads[self.SITE_HARD_TL_NAME] = site_hard_results
            metrics.extend(selective_results['metrics'])
            metrics.extend(site_hard_results['metrics'])

            if persist_artifacts:
                ModelEvaluator.save_table(selective_results['selector_search'], result_table_path('selector_search.csv'))
                ModelEvaluator.save_table(selective_results['selector_by_site'], result_table_path('selector_by_site.csv'))
                ModelEvaluator.save_table(selective_results['selector_allocation'], result_table_path('selector_allocation.csv'))
                ModelEvaluator.save_table(selective_results['selector_feature_importance'], result_table_path('selector_feature_importance.csv'))
                ModelEvaluator.save_table(site_hard_results['selector_by_site'], result_table_path('site_hard_selector_by_site.csv'))

        result_frame = ModelEvaluator.to_frame(metrics)
        summary_long = ModelEvaluator.extract_split_summary(result_frame)
        summary_wide = ModelEvaluator.wide_summary(summary_long)
        gain_df = ModelEvaluator.gain_vs_baseline(summary_long, baseline=self.NO_TL_NAME)
        site_metric_long, site_metric_wide = ModelEvaluator.site_level_metrics(model_payloads)
        best_model_by_site = ModelEvaluator.best_model_by_site(site_metric_long)

        self.results['summary_long'] = summary_long
        self.results['summary_wide'] = summary_wide
        self.results['gain_vs_baseline'] = gain_df
        self.results['site_metric_long'] = site_metric_long
        self.results['site_metric_wide'] = site_metric_wide
        self.results['best_model_by_site'] = best_model_by_site
        self.results['publication_core_comparison'] = self.build_publication_core_comparison(summary_wide)

        error_detail = self.build_prediction_error_detail(model_payloads, split='test')
        error_tables = self.build_error_analysis_tables(error_detail, site_metric_wide)
        self.results['test_error_detail'] = error_detail
        self.results.update(error_tables)

        if persist_artifacts:
            if not error_detail.empty:
                ModelEvaluator.save_table(error_detail, result_table_path('test_error_detail.csv'))
            for table_name, table_df in error_tables.items():
                if isinstance(table_df, pd.DataFrame) and not table_df.empty:
                    ModelEvaluator.save_table(table_df, result_table_path(f'{table_name}.csv'))

        if persist_artifacts:
            ModelEvaluator.save_table(summary_long, result_table_path('summary_long.csv'))
            ModelEvaluator.save_table(summary_wide, result_table_path('summary_wide.csv'))
            ModelEvaluator.save_table(gain_df, result_table_path('gain_vs_no_tl.csv'))
            ModelEvaluator.save_table(site_metric_long, result_table_path('site_metric_long.csv'))
            ModelEvaluator.save_table(site_metric_wide, result_table_path('site_metric_wide.csv'))
            ModelEvaluator.save_table(best_model_by_site, result_table_path('best_model_by_site.csv'))
            if not self.results['publication_core_comparison'].empty:
                ModelEvaluator.save_table(
                    self.results['publication_core_comparison'],
                    result_table_path('publication_core_comparison.csv'),
                )

        figure_paths = []
        if persist_artifacts:
            figure_paths.extend(
                [
                    ModelEvaluator.plot_model_performance(summary_long, self.figure_dir),
                    DomainAnalyzer.plot_similarity_distribution(self.tables['best_matches'], self.figure_dir),
                ]
            )

        if run_selective_tl:
            site_transfer_effects, transfer_effect_summary = self.build_site_transfer_effects(site_metric_wide)
            self.results['site_transfer_effects'] = site_transfer_effects
            self.results['transfer_effect_summary'] = transfer_effect_summary

            if persist_artifacts:
                ModelEvaluator.save_table(site_transfer_effects, result_table_path('site_transfer_effects.csv'))
                ModelEvaluator.save_table(transfer_effect_summary, result_table_path('transfer_effect_summary.csv'))

                selector_plot = self.plot_selector_candidates(self.results['selective_tl']['selector_search'])
                if selector_plot is not None:
                    figure_paths.append(selector_plot)
                figure_paths.append(self.plot_similarity_vs_gain(site_transfer_effects))
                no_tl_vs_selective = self.plot_no_tl_vs_selective(site_metric_wide)
                if no_tl_vs_selective is not None:
                    figure_paths.append(no_tl_vs_selective)

            cohort_definitions, cohort_assignments = self.build_analysis_cohorts()
            cohort_site_summary = self.build_cohort_site_summary(site_metric_wide, cohort_assignments)
            cohort_pooled_summary = self.build_cohort_pooled_summary(model_payloads, cohort_assignments, split='test')

            self.results['cohort_definitions'] = cohort_definitions
            self.results['cohort_assignments'] = cohort_assignments
            self.results['cohort_site_summary'] = cohort_site_summary
            self.results['cohort_pooled_summary'] = cohort_pooled_summary

            if persist_artifacts:
                ModelEvaluator.save_table(cohort_definitions, result_table_path('cohort_definitions.csv'))
                ModelEvaluator.save_table(cohort_assignments, result_table_path('cohort_assignments.csv'))
                ModelEvaluator.save_table(cohort_site_summary, result_table_path('cohort_site_summary.csv'))
                ModelEvaluator.save_table(cohort_pooled_summary, result_table_path('cohort_pooled_summary.csv'))
                cohort_plot = self.plot_cohort_site_summary(cohort_site_summary)
                if cohort_plot is not None:
                    figure_paths.append(cohort_plot)

            negative_transfer_site, negative_transfer_summary, negative_transfer_bins = self.build_negative_transfer_diagnostics(
                model_payloads=model_payloads,
                site_metric_wide=site_metric_wide,
                split='test',
            )
            self.results['negative_transfer_site_summary'] = negative_transfer_site
            self.results['negative_transfer_summary'] = negative_transfer_summary
            self.results['negative_transfer_similarity_bins'] = negative_transfer_bins

            if persist_artifacts:
                if not negative_transfer_site.empty:
                    ModelEvaluator.save_table(negative_transfer_site, result_table_path('negative_transfer_site_summary.csv'))
                if not negative_transfer_summary.empty:
                    ModelEvaluator.save_table(negative_transfer_summary, result_table_path('negative_transfer_summary.csv'))
                if not negative_transfer_bins.empty:
                    ModelEvaluator.save_table(negative_transfer_bins, result_table_path('negative_transfer_similarity_bins.csv'))
                    negative_transfer_plot = self.plot_negative_transfer_bins(negative_transfer_bins)
                    if negative_transfer_plot is not None:
                        figure_paths.append(negative_transfer_plot)

            if run_selector_ablation:
                selector_ablation_summary = self.run_selector_ablations()
                self.results['selector_ablation_summary'] = selector_ablation_summary
                if persist_artifacts and not selector_ablation_summary.empty:
                    ModelEvaluator.save_table(selector_ablation_summary, result_table_path('selector_ablation_summary.csv'))
                    selector_ablation_plot = self.plot_selector_ablation(selector_ablation_summary)
                    if selector_ablation_plot is not None:
                        figure_paths.append(selector_ablation_plot)

        if run_significance and run_selective_tl:
            significance_df = self.compute_significance_tests(model_payloads, split='test')
            bootstrap_df = self.compute_cluster_bootstrap_tests(model_payloads, split='test')
            self.results['paired_significance_tests'] = significance_df
            self.results['cluster_bootstrap_significance'] = bootstrap_df
            if persist_artifacts and not significance_df.empty:
                ModelEvaluator.save_table(significance_df, result_table_path('paired_significance_tests.csv'))
            if persist_artifacts and not bootstrap_df.empty:
                ModelEvaluator.save_table(bootstrap_df, result_table_path('cluster_bootstrap_significance.csv'))

        if run_label_budget:
            label_budget_detail, label_budget_summary = self.run_label_budget_evaluation(
                budgets=label_budget_windows,
                run_rf=run_rf,
                run_selective_tl=run_selective_tl,
                run_baseline=run_baseline,
            )
            self.results['label_budget_detail'] = label_budget_detail
            self.results['label_budget_summary'] = label_budget_summary
            if persist_artifacts:
                ModelEvaluator.save_table(label_budget_detail, result_table_path('label_budget_detail.csv'))
                ModelEvaluator.save_table(label_budget_summary, result_table_path('label_budget_summary.csv'))
                label_budget_plot = self.plot_label_budget_curve(label_budget_summary)
                if label_budget_plot is not None:
                    figure_paths.append(label_budget_plot)

        if run_seed_stability:
            stability_seeds = [11, 23, 42, 67, 89] if stability_seeds is None else [int(seed) for seed in stability_seeds]
            stability_detail, stability_summary = self.run_seed_stability(
                seeds=stability_seeds,
                run_rf=run_rf,
                run_selective_tl=run_selective_tl,
                run_baseline=run_baseline,
            )
            self.results['seed_stability_detail'] = stability_detail
            self.results['seed_stability_summary'] = stability_summary
            if persist_artifacts:
                ModelEvaluator.save_table(stability_detail, result_table_path('seed_stability_detail.csv'))
                ModelEvaluator.save_table(stability_summary, result_table_path('seed_stability_summary.csv'))
                figure_paths.append(ModelEvaluator.plot_seed_stability(stability_summary, self.figure_dir))

        if run_rolling_temporal:
            rolling_folds, rolling_detail, rolling_summary = self.run_rolling_temporal_evaluation(
                fold_count=rolling_fold_count,
                train_months=rolling_train_months,
                val_months=rolling_val_months,
                cal_months=rolling_cal_months,
                test_months=rolling_test_months,
                run_rf=run_rf,
                run_selective_tl=run_selective_tl,
                run_baseline=run_baseline,
            )
            self.results['rolling_temporal_folds'] = rolling_folds
            self.results['rolling_temporal_detail'] = rolling_detail
            self.results['rolling_temporal_summary'] = rolling_summary
            if persist_artifacts:
                ModelEvaluator.save_table(rolling_folds, result_table_path('rolling_temporal_folds.csv'))
                ModelEvaluator.save_table(rolling_detail, result_table_path('rolling_temporal_detail.csv'))
                ModelEvaluator.save_table(rolling_summary, result_table_path('rolling_temporal_summary.csv'))
                figure_paths.append(ModelEvaluator.plot_rolling_temporal_stability(rolling_summary, self.figure_dir))

        robustness_comparison = self.build_publication_robustness_comparison()
        self.results['publication_robustness_comparison'] = robustness_comparison
        self.results['figure_paths'] = [path for path in figure_paths if path is not None]
        if persist_artifacts and not robustness_comparison.empty:
            ModelEvaluator.save_table(
                robustness_comparison,
                result_table_path('publication_robustness_comparison.csv'),
            )

        summary_display = summary_wide[['model_family', 'rmse_test', 'mae_test', 'r2_test']].copy()
        numeric_cols = [col for col in summary_display.columns if col != 'model_family']
        summary_display[numeric_cols] = summary_display[numeric_cols].round(4)
        if display_outputs:
            display(summary_display)
            negative_transfer_display = self.results.get('negative_transfer_summary')
            if isinstance(negative_transfer_display, pd.DataFrame) and not negative_transfer_display.empty:
                display(negative_transfer_display.round(4))

            selector_ablation_display = self.results.get('selector_ablation_summary')
            if isinstance(selector_ablation_display, pd.DataFrame) and not selector_ablation_display.empty:
                display(
                    selector_ablation_display[
                        ['ablation_label', 'n_features', 'rmse_test', 'delta_rmse_vs_main']
                    ].round(4)
                )

            label_budget_display = self.results.get('label_budget_summary')
            if isinstance(label_budget_display, pd.DataFrame) and not label_budget_display.empty:
                display(
                    label_budget_display[
                        label_budget_display['model_family'].isin(
                            [self.SITE_HARD_TL_NAME, self.SELECTIVE_TL_NAME, self.ALWAYS_TL_NAME, self.NO_TL_NAME, 'Random Forest']
                        )
                    ][
                        [
                            'budget_label',
                            'model_family',
                            'mean_train_windows_per_site',
                            'rmse',
                            'delta_rmse_vs_no_tl_same_budget',
                            'delta_rmse_vs_always_tl_same_budget',
                        ]
                    ].round(4)
                )

            if not robustness_comparison.empty:
                display(robustness_comparison.round(4))
            print(f"Detailed tables saved to: {self.table_dir}")
            print(f"Figures saved to: {self.figure_dir}")

        return self.results

display(read_result_csv('feature_inventory.csv'))
read_result_csv('transfer_configuration.csv').round(3)

# Main Execution

Here I load the saved publication results and show the tables and figures that support my final comparison. If the saved results are missing, I can still run the full pipeline from this cell.

In [ ]:
def main(
    run_rf=True,
    run_selective_tl=True,
    run_baseline=True,
    run_significance=True,
    run_seed_stability=True,
    run_rolling_temporal=True,
    run_selector_ablation=True,
    run_label_budget=True,
    label_budget_windows=None,
    stability_seeds=None,
    rolling_fold_count=3,
    seed=42,
):
    implementation = LSTMTransferLearningImplementation(root=ROOT, sequence_length=9, seed=seed)
    results = implementation.run(
        run_rf=run_rf,
        run_selective_tl=run_selective_tl,
        run_baseline=run_baseline,
        run_significance=run_significance,
        run_seed_stability=run_seed_stability,
        run_rolling_temporal=run_rolling_temporal,
        run_selector_ablation=run_selector_ablation,
        run_label_budget=run_label_budget,
        label_budget_windows=label_budget_windows,
        stability_seeds=stability_seeds,
        rolling_fold_count=rolling_fold_count,
    )
    return results


if methodology_results_available():
    print('I load the saved methodology results from disk.')
    results = {'mode': 'loaded_saved_results'}
else:
    print('I could not find the saved results, so I run the full publication pipeline.')
    results = main(
        run_rf=True,
        run_selective_tl=True,
        run_baseline=True,
        run_significance=True,
        run_seed_stability=True,
        run_rolling_temporal=True,
        run_selector_ablation=True,
        run_label_budget=True,
        label_budget_windows=[12, 24, 48, 72],
        stability_seeds=[11, 23, 42, 67, 89],
        rolling_fold_count=3,
        seed=42,
    )

core_comparison = read_result_csv('publication_core_comparison.csv').round(3)
transfer_effects = read_result_csv('transfer_effect_summary.csv').round(3)
selector_ablation = read_result_csv('selector_ablation_summary.csv').round(3)
robustness = read_result_csv('publication_robustness_comparison.csv').round(3)

display(core_comparison)
display(transfer_effects)
display(selector_ablation[['ablation_label', 'rmse_test', 'delta_rmse_vs_main']].head())
display(show_saved_figure('model_performance_test.png', figsize=(8.5, 4.8)))
display(show_saved_figure('selector_ablation_test_rmse.png', figsize=(8.5, 4.8)))
display(show_saved_figure('rolling_temporal_test_rmse.png', figsize=(8.5, 4.8)))
display(show_saved_figure('label_budget_curve.png', figsize=(8.5, 4.8)))
robustness
